# Theory of Time Results v4: Compact Benchmark Scoreboard and Diagnostics

This notebook extends analysis v3.5 with targeted construct-validity corrections and sensitivity analyses. It otherwise preserves the existing Track 1 metric definitions, Track 2 analysis-layer score construction, three-domain scoreboard, matched component gaps, domain-specific prior analyses, harmonized cross-track diagnostics, and failure-stage taxonomy.

The main benchmark uses ordinary `active` runs: binned Discovery for Tracks 1A/1B/1C and continuous-time Track 2. Scores are stored on the 0–1 scale and displayed as percentages. Missing values remain missing when a metric has no meaningful denominator.

Two analyses from the v3 design are intentionally not implemented. Track 2 absence updates do not record claim-time confidence, and no Track 2 periodic-probe result files currently exist. The notebook therefore contains 15 sequentially numbered panels. Analysis v4 also distinguishes factual absence accuracy from claim-time coverage support, uses symmetric unresolved-bin scoring in the Track 1A/1B access comparison, and treats all stage orderings and thresholds as descriptive diagnostics.


## Setup and shared data assets

The setup is retained from v3.5. Analysis v4 adds item-cluster rank uncertainty and declared threshold-sensitivity scenarios without changing the active benchmark population. A run remains evaluable when its applicable score can be calculated; low evidence acquisition and invalid protocol behavior remain benchmark failures rather than exclusion criteria. Overall ranking requires complete evaluable support for every expected active unit.

Paired contrasts report deterministic percentile-bootstrap 95% confidence intervals using 10,000 cluster resamples. The resampling unit is the episode for Tracks 1A/1B, the base `conversation_id` for Track 1C, and the `reel_id` for Track 2. These intervals quantify sensitivity to benchmark items; they do not estimate run-to-run model stochasticity. Eligible contrast heatmaps retain paired cluster-level sign-flip tests at Holm-adjusted `p < 0.05`, but Panels 4 and 5 explicitly treat those tests as uninformative: with only ten Track 1A/1B episode clusters, the minimum attainable exact two-sided p-value cannot survive their 80-cell and approximately 100-cell correction families. Effect sizes and clustered intervals, not the absence of significance markers, carry the interpretation. Panel 1 adds paired item-cluster bootstrap rank uncertainty; it still cannot quantify generation-to-generation model variability.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from matplotlib import ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display as ipython_display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 180)

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
sys.path.insert(0, str(repo_root))
from analysis.frozen_inputs import frozen_input_paths, verify_inputs
verify_inputs(repo_root)
ANALYSIS_OUTPUT_DIR = repo_root / "analysis" / "outputs" / "paper_plot_v5" / "supplement"


def panel_output_dir(panel):
    output_dir = ANALYSIS_OUTPUT_DIR / panel
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def save_panel_table(frame, panel, filename):
    assert filename.startswith("table_") and filename.endswith(".csv")
    frame.to_csv(panel_output_dir(panel) / filename, index=False)



def rename_v5_figure_text(figure):
    """Translate legacy display labels once; never change analysis identifiers."""
    import re
    from matplotlib.text import Text

    exact = {
        "Tracks 1A and 1B mean": "Track 1 (setting mean)",
        "Tracks 1A/1B mean": "Track 1\n(setting mean)",
        "(a) Controlled acoustic temporal mapping": "(a) Track 1: Controlled Acoustic\nTemporal Mapping",
        "(b) Controlled conversational belief profile": "(b) Track 2: Controlled Conversational\nBelief Updating",
        "Track 1": "Tracks 1 and 2",  # Combined 1A/1B/1C uncertainty heatmap.
        "1A": "Sparse", "1B": "Concurrent", "1C": "Track 2",
        "1a": "Sparse", "1b": "Concurrent", "1c": "Track 2",
        "Correct\ntarget\nstatuses": "Target-status\ncorrectness",
        "Complete\nbelief\nreport": "Belief report\ncompleteness",
        "Found\nsupport ↑": "Presence\nsupport ↑",
        "Absence\nsupported ↑": "Absence\nsupport ↑",
        "Calibration ↑": "Confidence\nconsistency ↑",
        "Bias (pp)": "Known–unresolved\nscore difference (pp)",
        "Family": "Task", "family": "task",
        "Budget used": "Budget use",
        "Temporal belief": "Temporal belief score",
        "Master score": "Composite score",
        "Evidence\nreached": "Evidence\nreach",
        "Targets\ncontacted": "Required\ntargets\nacquired",
        "Ledger\ntarget": "Target-\nstatus\ncorrectness",
        "Ledger\ncomplete": "Belief\nreport\ncompleteness",
        "Task\ncorrect": "Task\ncorrectness",
        "Strict\ncomplete": "Strict\ncompletion",
        "Completion rate (%)": "Score / rate (%)",
        "Temporal instruction\nsequence": "Temporal instruction\nsequences",
        "Delayed answer and\ndeferred resolution": "Delayed answers and\ndeferred resolution",
        "Topic switch\nwith distractors": "Topic switches\nwith distractors",
        "acquisition-limited": "Acquisition-limited\n(reach < 0.5; belief < 0.6)",
        "belief-limited": "Belief-limited\n(reach ≥ 0.5; belief < 0.6)",
        "robust": "Robust\n(reach ≥ 0.5; belief ≥ 0.6)",
        "inspect priors/metric mismatch": "High belief / low reach\n(reach < 0.5; belief ≥ 0.6)",
    }
    replacements = {
        "Track 1 known-minus-unknown bias": "Tracks 1 and 2: known–unresolved score difference",
        "Track 1 contrast cells": "Tracks 1 and 2 contrast cells",
        "Known - unknown (pp)": "Known–unresolved score difference (pp)",
        "Track 1A": "Track 1 — Sparse setting",
        "Track 1B": "Track 1 — Concurrent setting",
        "Track 1C": "Track 2",
        "Track 2A.1": "Task 3.1", "Track 2A.2": "Task 3.2",
        "Track 2A.3": "Task 3.3", "Track 2B.1": "Task 3.4",
        "Track 2B.2": "Task 3.5",
        "Track 2A": "Tasks 3.1–3.3", "Track 2B": "Tasks 3.4–3.5",
        "Track 2": "Track 3",
        "2A.1": "Task 3.1", "2A.2": "Task 3.2", "2A.3": "Task 3.3",
        "2B.1": "Task 3.4", "2B.2": "Task 3.5",
        "2A": "Tasks 3.1–3.3", "2B": "Tasks 3.4–3.5",
        "CAM": "Track 1: Controlled Acoustic Temporal Mapping",
        "CBU": "Track 2: Controlled Conversational Belief Updating",
        "LCN": "Track 3: Long-Horizon Conversational Navigation",
        "family-balanced": "task-balanced", "Task families": "Tasks",
        "task families": "tasks", "by family": "by task",
        "Family and variant": "Task and variant",
        "Domain scores": "Track scores", "cross-domain": "cross-track",
        "Cross-domain": "Cross-track", "capability domain": "track",
        "418 tasks": "418 task instances",
        "Full context": "Full-context", "full context": "full-context",
        "Full-context access": "Full-context condition",
        "Active access": "Active condition", "active access": "active condition",
        "Information regimes": "Prior-information levels",
        "information regimes": "prior-information levels",
        "Discovery condition": "Discovery prior-information level",
        "Discovery regime": "Discovery prior-information level",
        "three-regime": "three-prior-level",
        "Belief ledger": "Temporal belief report", "belief ledger": "belief report",
        "Ledger target": "Target-status correctness", "Ledger complete": "Belief report completeness",
        "ledger": "belief report", "Reel": "Episode", "reel": "episode",
        "Scene card": "Scene description", "scene card": "scene description",
        "Targets contacted": "Required targets heard",
        "Required targets contacted": "Required targets heard",
        "required-target contact": "whether all required targets were heard",
        "Descriptive bias": "Known–unresolved score difference",
        "known-minus-unknown bias": "known–unresolved score difference",
        "Mean semantic belief score (%)": "Mean semantic state score (%)",
        "Target evidence found": "Complete target-evidence acquisition",
        "Stale answer rate": "Evidence-conditioned stale-answer rate",
        "Repair prompt": "Correction prompt", "repair prompt": "correction prompt",
        "post-repair validity": "validity after permitted correction",
        "Correctness of required structured fields": "Correctness of required answer fields",
    }
    pattern = re.compile(r"(?<![\w])(?:" + "|".join(re.escape(key) for key in sorted(replacements, key=len, reverse=True)) + r")(?![\w])")
    def translate(label):
        return exact[label] if label in exact else pattern.sub(lambda match: replacements[match.group()], label)

    # Update fixed tick labels too: a draw would otherwise restore the old formatter text.
    for axis in figure.axes:
        for coordinate in (axis.xaxis, axis.yaxis):
            labels = coordinate.get_ticklabels()
            current = [label.get_text() for label in labels]
            renamed = [label if getattr(artist, "_v5_label_normalized", False) else translate(label)
                       for artist, label in zip(labels, current)]
            if current in (["1A", "1B", "1C"], ["1a", "1b", "1c"]):
                renamed = ["Track 1\nSparse", "Track 1\nConcurrent", "Track 2"]
            if renamed != current:
                coordinate.set_ticks(coordinate.get_ticklocs(), labels=renamed)
            for artist in coordinate.get_ticklabels():
                artist._v5_label_normalized = True
    for artist in figure.findobj(match=Text):
        if not getattr(artist, "_v5_font_enlarged", False):
            artist.set_fontsize(getattr(artist, "_v5_font_scale", 1.15) * artist.get_fontsize())
            artist._v5_font_enlarged = True
        if getattr(artist, "_v5_label_normalized", False):
            continue
        label = artist.get_text()
        artist.set_text(translate(label))
        artist._v5_label_normalized = True


def save_panel_figure(figure, panel, filename):
    assert filename.startswith("plot_") and filename.endswith(".png")
    rename_v5_figure_text(figure)
    figure.savefig(panel_output_dir(panel) / filename, format="png", dpi=600, bbox_inches="tight", facecolor="white")

TRACKS = ["1a", "1b", "1c"]
MAIN_TIME_MODE = "binned"
MAIN_REGIME = "Discovery"
DIAGNOSTIC_THRESHOLD_SCENARIOS = {
    "lenient": {"protocol": 0.80, "evidence": 0.40, "low_score": 0.50, "cost": 0.45, "drift": 0.30, "stale": 0.40, "high_budget": 0.90, "budget_reach": 0.60},
    "declared": {"protocol": 0.90, "evidence": 0.50, "low_score": 0.60, "cost": 0.35, "drift": 0.20, "stale": 0.30, "high_budget": 0.80, "budget_reach": 0.70},
    "strict": {"protocol": 0.95, "evidence": 0.60, "low_score": 0.70, "cost": 0.25, "drift": 0.10, "stale": 0.20, "high_budget": 0.70, "budget_reach": 0.80},
}
DECLARED_THRESHOLDS = DIAGNOSTIC_THRESHOLD_SCENARIOS["declared"]
PROTOCOL_THRESHOLD = DECLARED_THRESHOLDS["protocol"]
PROBE_THRESHOLD = 0.90
EVIDENCE_THRESHOLD = DECLARED_THRESHOLDS["evidence"]
LOW_SCORE_THRESHOLD = DECLARED_THRESHOLDS["low_score"]

MODEL_NAME_OVERRIDES = {
    "audio_flamingo_next": "Audio Flamingo Next",
    "audio_flamingo_next_think": "Audio Flamingo Next Think",
    "gemini_2_flash": "Gemini 2 Flash",
    "gemini_3_flash": "Gemini 3 Flash",
    "gpt_audio": "GPT Audio",
    "gpt_audio_mini": "GPT Audio Mini",
    "kimi_audio": "Kimi Audio",
    "mimo_audio": "Mimo Audio",
    "moss_audio": "MOSS Audio",
    "qwen25_omni": "Qwen2.5 Omni",
}



def normalize_track_id(value, path=None):
    text = "" if pd.isna(value) else str(value).lower()
    path_text = "" if path is None else str(path).lower()
    for track in TRACKS:
        if text in {track, f"track_{track}", f"track{track}"}:
            return track
        if track in text or f"track_{track}" in path_text or f"track{track}" in path_text:
            return track
    return text or "unknown"


def condition_family(condition):
    condition = str(condition)
    if condition == "full_context":
        return "full_context"
    if condition.startswith("active_same_session"):
        return "active_same_session"
    if condition == "active":
        return "active"
    return condition


def score_col(track, family, component="master"):
    suffix = {
        "active": "active",
        "active_same_session": "active_same_session",
        "full_context": "full_context",
    }.get(family)
    if suffix is None or track not in TRACKS:
        return None
    return f"track{track}_{suffix}_{component}_score"


def coalesce_columns(frame, names):
    present = [name for name in names if name in frame.columns]
    if not present:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return frame[present].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]


def mean_columns(frame, names):
    present = [name for name in names if name in frame.columns]
    if not present:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return frame[present].apply(pd.to_numeric, errors="coerce").mean(axis=1, skipna=True)


def divide_or_nan(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator / denominator.where(denominator > 0)


def normalize_unit_score(series):
    series = pd.to_numeric(series, errors="coerce")
    finite = series.dropna()
    assert finite.between(0.0, 1.0).all(), "Expected a score on the 0-1 scale"
    return series


def normalize_master_score(series):
    series = pd.to_numeric(series, errors="coerce")
    finite = series.dropna()
    assert finite.between(0.0, 100.0).all(), "Expected a master score on the 0-100 scale"
    return series / 100.0


def metric_file_track_ids(csv_path):
    columns = pd.read_csv(csv_path, nrows=0).columns
    if "track_id" not in columns:
        return set()
    values = pd.read_csv(csv_path, usecols=["track_id"])["track_id"].dropna().unique()
    return {normalize_track_id(value, csv_path) for value in values} & set(TRACKS)


def discover_episode_metric_files(root):
    return frozen_input_paths(root, "track1_metric_files")


def load_episode_metrics(root):
    pieces = []
    for csv_path in discover_episode_metric_files(root):
        model_root = csv_path.relative_to(root).parts[0]
        frame = pd.read_csv(csv_path)
        frame["model_root"] = model_root
        frame["model"] = MODEL_NAME_OVERRIDES.get(model_root, model_root)
        frame["result_set"] = csv_path.relative_to(root / model_root).parts[0]
        frame["result_root"] = str(csv_path.parent.relative_to(root))
        frame["source_csv"] = str(csv_path.relative_to(root))
        frame["track"] = [normalize_track_id(v, csv_path) for v in frame.get("track_id", pd.Series([None] * len(frame)))]
        frame["condition_family"] = frame["condition"].map(condition_family)
        if "time_mode" not in frame.columns:
            frame["time_mode"] = "continuous" if "results_continuous" in str(csv_path) else "binned"
        pieces.append(frame)
    metrics = pd.concat(pieces, ignore_index=True, sort=False)
    metrics = metrics[metrics["track"].isin(TRACKS)].copy()
    assert not metrics.empty, "Loaded files, but no Track 1a/1b/1c rows were found"
    logical_row_key = ["model_root", "time_mode", "track", "condition", "regime", "episode_id"]
    duplicate_rows = metrics.duplicated(logical_row_key, keep=False)
    assert not duplicate_rows.any(), (
        "Multiple metric rows found for logical episodes:\n"
        + metrics.loc[duplicate_rows, logical_row_key + ["source_csv"]]
        .sort_values(logical_row_key)
        .to_string(index=False)
    )
    return metrics


episode_metrics = load_episode_metrics(repo_root)

score_values = []
answer_values = []
protocol_values = []
exploration_values = []
for row in episode_metrics.itertuples(index=False):
    row_dict = row._asdict()
    columns = episode_metrics.columns
    values = []
    for component in ["master", "answer", "protocol", "exploration"]:
        column = score_col(row_dict["track"], row_dict["condition_family"], component)
        values.append(row_dict[column] if column in columns and pd.notna(row_dict.get(column)) else np.nan)
    score_values.append(values[0])
    answer_values.append(values[1])
    protocol_values.append(values[2])
    exploration_values.append(values[3])

episode_metrics["master_score"] = score_values
episode_metrics["answer_score"] = answer_values
episode_metrics["protocol_component_score"] = protocol_values
episode_metrics["exploration_component_score"] = exploration_values
episode_metrics["master_score"] = normalize_master_score(episode_metrics["master_score"])
for score_name in ["answer_score", "protocol_component_score", "exploration_component_score"]:
    episode_metrics[score_name] = normalize_unit_score(episode_metrics[score_name])

episode_metrics["protocol_valid_score"] = mean_columns(
    episode_metrics,
    ["action_format_valid_rate", "action_semantic_valid_rate"],
)
episode_metrics["probe_valid_score"] = coalesce_columns(episode_metrics, ["probe_valid_rate"])

event_acquisition = mean_columns(
    episode_metrics,
    ["event_coverage_rate", "event_bin_coverage_rate", "event_time_coverage_rate"],
)
track1c_acquisition = mean_columns(
    episode_metrics,
    ["target_evidence_sentence_recall", "sentence_coverage_rate"],
)
episode_metrics["acquisition_score"] = event_acquisition.where(episode_metrics["track"] != "1c", track1c_acquisition)

episode_metrics["bin_map_score"] = coalesce_columns(episode_metrics, ["belief_bin_score_mean", "belief_time_score_mean"])
episode_metrics["order_score"] = coalesce_columns(episode_metrics, ["belief_pairwise_accuracy", "core_behavior_pairwise_accuracy"])
episode_metrics["localization_score"] = coalesce_columns(
    episode_metrics,
    ["belief_localization_mean_iou", "belief_localization_exact_250ms", "belief_localization_exact", "track1c_timing_localization_iou"],
)
episode_metrics["gap_score"] = coalesce_columns(episode_metrics, ["belief_gap_accuracy", "core_behavior_gap_accuracy"])
episode_metrics["overlap_score"] = mean_columns(
    episode_metrics,
    ["belief_overlap_pair_f1", "belief_overlap_interval_iou", "core_behavior_overlap_accuracy", "core_behavior_overlap_iou"],
)
episode_metrics["semantic_score"] = mean_columns(
    episode_metrics,
    ["track1c_semantic_accuracy", "track1c_content_accuracy", "track1c_generic_accuracy", "track1c_category_accuracy"],
)
episode_metrics["speaker_score"] = coalesce_columns(
    episode_metrics,
    ["track1c_speaker_raster_token_accuracy", "track1c_speaker_raster_exact"],
)
episode_metrics["update_score"] = coalesce_columns(episode_metrics, ["track1c_update_success_rate"])

belief_1ab = mean_columns(
    episode_metrics,
    ["bin_map_score", "order_score", "localization_score", "gap_score", "overlap_score"],
)
belief_1c = mean_columns(
    episode_metrics,
    ["semantic_score", "localization_score", "speaker_score", "update_score"],
)
episode_metrics["belief_score"] = belief_1ab.where(episode_metrics["track"] != "1c", belief_1c)

uncertainty_1ab = mean_columns(
    episode_metrics,
    ["uncertainty_pairwise_accuracy", "uncertainty_localization_accuracy", "uncertainty_gap_accuracy", "uncertainty_overlap_accuracy"],
)
episode_metrics["unknown_score"] = coalesce_columns(episode_metrics, ["track1c_unknown_accuracy"]).where(
    episode_metrics["track"] == "1c", uncertainty_1ab
)
episode_metrics["known_score"] = coalesce_columns(episode_metrics, ["track1c_non_unknown_accuracy"]).where(
    episode_metrics["track"] == "1c", episode_metrics["belief_score"]
)
episode_metrics["uncertainty_score"] = mean_columns(episode_metrics, ["unknown_score", "known_score"])

useful_count = coalesce_columns(
    episode_metrics,
    ["event_bin_coverage_count", "event_coverage_count", "event_time_coverage_seconds", "target_evidence_sentence_recall"],
)
listened_total = coalesce_columns(episode_metrics, ["listened_bin_total", "listened_seconds_total"])
episode_metrics["evidence_yield"] = divide_or_nan(useful_count, listened_total)
episode_metrics["repeat_waste"] = divide_or_nan(
    coalesce_columns(episode_metrics, ["repeated_bin_total", "repeated_seconds_total"]),
    listened_total,
)
episode_metrics["silent_waste"] = coalesce_columns(episode_metrics, ["silent_bin_coverage_rate", "silent_time_coverage_rate"])
episode_metrics["redundant_query_rate"] = divide_or_nan(
    coalesce_columns(episode_metrics, ["redundant_query_count"]),
    coalesce_columns(episode_metrics, ["query_count"]),
)
episode_metrics["efficiency_score"] = mean_columns(episode_metrics, ["evidence_yield"]) - mean_columns(
    episode_metrics,
    ["repeat_waste", "silent_waste", "redundant_query_rate"],
)

episode_metrics["drift_score"] = np.nan
episode_metrics["revision_score"] = coalesce_columns(episode_metrics, ["track1c_update_success_rate"])

diagnostic_columns = [
    "master_score",
    "answer_score",
    "protocol_component_score",
    "exploration_component_score",
    "protocol_valid_score",
    "probe_valid_score",
    "acquisition_score",
    "evidence_yield",
    "repeat_waste",
    "silent_waste",
    "redundant_query_rate",
    "efficiency_score",
    "belief_score",
    "bin_map_score",
    "order_score",
    "localization_score",
    "gap_score",
    "overlap_score",
    "semantic_score",
    "speaker_score",
    "update_score",
    "unknown_score",
    "known_score",
    "uncertainty_score",
    "revision_score",
    "drift_score",
]


def main_slice(frame, conditions=None):
    out = frame[(frame["time_mode"] == MAIN_TIME_MODE) & frame["track"].isin(TRACKS)].copy()
    if MAIN_REGIME is not None and MAIN_REGIME in set(out["regime"].dropna()):
        out = out[out["regime"] == MAIN_REGIME].copy()
    if conditions is not None:
        out = out[out["condition_family"].isin(conditions)].copy()
    return out


def aggregate(frame, group_cols, value_cols=diagnostic_columns):
    cols = [col for col in value_cols if col in frame.columns]
    return frame.groupby(group_cols, dropna=False)[cols].mean().reset_index()


def percent_table(frame, cols):
    out = frame.copy()
    for col in cols:
        if col in out.columns:
            out[col] = (100 * out[col]).round(1)
    return out


PAIRED_CI_LEVEL = 0.95
PAIRED_CI_BOOTSTRAP_REPLICATES = 10_000
PAIRED_CI_RANDOM_SEED = 20260724
PAIRED_SIGN_FLIP_REPLICATES = 50_000
PAIRED_SIGN_FLIP_EXACT_MAX_CLUSTERS = 16
PAIRED_SIGN_FLIP_RANDOM_SEED = 20260725
SIGNIFICANCE_ALPHA = 0.05
_SIGN_FLIP_MATRIX_CACHE = {}


def clustered_mean_ci(values, clusters):
    paired = pd.DataFrame({
        "value": pd.to_numeric(values, errors="coerce"),
        "cluster": pd.Series(clusters, index=getattr(values, "index", None)),
    }).dropna(subset=["value", "cluster"])
    if paired.empty:
        return np.nan, np.nan, np.nan, 0, 0
    cluster_totals = paired.groupby("cluster", sort=True)["value"].agg(["sum", "count"])
    estimate = paired["value"].mean()
    cluster_count = len(cluster_totals)
    if cluster_count < 2:
        return estimate, np.nan, np.nan, cluster_count, len(paired)
    rng = np.random.default_rng(PAIRED_CI_RANDOM_SEED)
    sampled = rng.integers(0, cluster_count, size=(PAIRED_CI_BOOTSTRAP_REPLICATES, cluster_count))
    bootstrap_means = (
        cluster_totals["sum"].to_numpy()[sampled].sum(axis=1)
        / cluster_totals["count"].to_numpy()[sampled].sum(axis=1)
    )
    alpha = 1.0 - PAIRED_CI_LEVEL
    lower, upper = np.quantile(bootstrap_means, [alpha / 2.0, 1.0 - alpha / 2.0])
    return estimate, lower, upper, cluster_count, len(paired)


def cluster_sign_flip_p_value(values, clusters):
    paired = pd.DataFrame({
        "value": pd.to_numeric(values, errors="coerce"),
        "cluster": pd.Series(clusters, index=getattr(values, "index", None)),
    }).dropna(subset=["value", "cluster"])
    cluster_sums = paired.groupby("cluster", sort=True)["value"].sum().to_numpy(dtype=float)
    cluster_count = len(cluster_sums)
    if cluster_count == 0:
        return np.nan
    observed = abs(cluster_sums.sum())
    if np.isclose(observed, 0.0):
        return 1.0
    if cluster_count not in _SIGN_FLIP_MATRIX_CACHE:
        if cluster_count <= PAIRED_SIGN_FLIP_EXACT_MAX_CLUSTERS:
            codes = np.arange(1 << cluster_count, dtype=np.uint64)[:, None]
            bits = (codes >> np.arange(cluster_count, dtype=np.uint64)) & 1
            signs = (1 - 2 * bits.astype(np.int8)).astype(np.int8)
            exact = True
        else:
            rng = np.random.default_rng(PAIRED_SIGN_FLIP_RANDOM_SEED + cluster_count)
            signs = (2 * rng.integers(0, 2, size=(PAIRED_SIGN_FLIP_REPLICATES, cluster_count), dtype=np.int8) - 1).astype(np.int8)
            exact = False
        _SIGN_FLIP_MATRIX_CACHE[cluster_count] = (signs, exact)
    signs, exact = _SIGN_FLIP_MATRIX_CACHE[cluster_count]
    permuted = np.abs(signs @ cluster_sums)
    tolerance = np.finfo(float).eps * max(1.0, observed) * cluster_count
    extreme_count = np.count_nonzero(permuted >= observed - tolerance)
    return extreme_count / len(permuted) if exact else (extreme_count + 1) / (len(permuted) + 1)


def holm_adjusted_p_values(p_values):
    numeric = pd.to_numeric(p_values, errors="coerce")
    adjusted = pd.Series(np.nan, index=numeric.index, dtype="float64")
    valid = numeric.dropna()
    if valid.empty:
        return adjusted
    order = np.argsort(valid.to_numpy())
    ordered = valid.to_numpy()[order]
    scaled = (len(ordered) - np.arange(len(ordered))) * ordered
    ordered_adjusted = np.minimum(1.0, np.maximum.accumulate(scaled))
    adjusted.loc[valid.index[order]] = ordered_adjusted
    return adjusted


def track1_paired_cluster_id(frame):
    clusters = frame["episode_id"].astype("string").copy()
    track1c = frame["track"].eq("1c")
    assert frame.loc[track1c, "conversation_id"].notna().all(), "Track 1C paired CIs require conversation_id clusters"
    clusters.loc[track1c] = frame.loc[track1c, "conversation_id"].astype("string")
    return clusters


assert np.isclose(cluster_sign_flip_p_value(pd.Series(np.ones(6)), pd.Series(range(6))), 0.03125)
assert cluster_sign_flip_p_value(pd.Series(np.zeros(6)), pd.Series(range(6))) == 1.0
_holm_check = holm_adjusted_p_values(pd.Series([0.01, 0.03, 0.04, np.nan]))
assert np.allclose(_holm_check.iloc[:3], [0.03, 0.06, 0.06]) and pd.isna(_holm_check.iloc[3])


MODEL_LABEL_OVERRIDES = {
    "GPT Audio": "GPT",
    "GPT Audio Mini": "GPT-m",
    "Gemini 2 Flash": "Gem-2.5",
    "Gemini 2.5 Flash": "Gem-2.5",
    "Gemini 3 Flash": "Gem-3",
    "Audio Flamingo Next": "AFN",
    "Audio Flamingo Next Think": "AFN-T",
    "Kimi Audio": "Kimi",
    "Mimo Audio": "MiMo",
    "MOSS Audio": "MOSS",
    "Qwen2.5 Omni": "Qwen",
}

MODEL_COLORS = {
    "GPT": "#332288",
    "GPT-m": "#88CCEE",
    "Gem-2.5": "#DDCC77",
    "Gem-3": "#CC6677",
    "AFN": "#117733",
    "AFN-T": "#AA4499",
    "Kimi": "#44AA99",
    "MiMo": "#999933",
    "MOSS": "#882255",
    "Qwen": "#661100",
}


def short_model_label(model):
    model = str(model)
    return MODEL_LABEL_OVERRIDES.get(model, model)


def model_palette(models):
    palette = {model: MODEL_COLORS[short_model_label(model)] for model in models}
    assert len(palette) == len(set(models)), "Model palette keys must be unique"
    return palette


def display_model_labels(frame):
    out = frame.copy()
    for column in ["model", "Model"]:
        if column in out.columns:
            out[column] = out[column].map(short_model_label)
    if isinstance(out.index, pd.MultiIndex):
        arrays = [
            [short_model_label(value) for value in out.index.get_level_values(level)]
            if name in {"model", "Model"} else out.index.get_level_values(level)
            for level, name in enumerate(out.index.names)
        ]
        out.index = pd.MultiIndex.from_arrays(arrays, names=out.index.names)
    elif out.index.name in {"model", "Model"}:
        out.index = out.index.map(short_model_label)
    return out


def display(value):
    if isinstance(value, pd.DataFrame):
        value = display_model_labels(value)
    return ipython_display(value)


def add_master_score_size_legend(ax, anchor=(1.02, 1.0)):
    handles = []
    for score in [0.0, 0.5, 1.0]:
        handles.append(
            plt.Line2D(
                [],
                [],
                marker="o",
                linestyle="",
                markersize=np.sqrt(70 + 180 * score),
                markerfacecolor="lightgray",
                markeredgecolor="gray",
                label=f"{score:.0%}",
            )
        )
    ax.legend(handles=handles, title="Point size\nmaster score", loc="upper left", bbox_to_anchor=anchor, frameon=True)


def add_ideal_star(ax, x, y):
    ax.scatter(
        [x],
        [y],
        marker="*",
        s=90,
        color="red",
        edgecolor="white",
        linewidth=0.7,
        zorder=20,
        clip_on=False,
    )


def add_ideal_star_legend(grid, label="Ideal reference", next_section_label=None):
    assert grid._legend is not None, "An existing scatter legend is required"
    legend = grid._legend
    handles = list(legend.legend_handles)
    labels = [text.get_text() for text in legend.texts]
    assert label not in labels, f"Duplicate ideal-reference legend label: {label}"
    if next_section_label is None:
        insert_at = len(labels)
    else:
        assert next_section_label in labels, f"Missing legend section: {next_section_label}"
        insert_at = labels.index(next_section_label)
    handle = plt.Line2D(
        [], [], marker="*", linestyle="", markersize=10,
        markerfacecolor="red", markeredgecolor="white", label=label,
    )
    handles.insert(insert_at, handle)
    labels.insert(insert_at, label)
    title = legend.get_title().get_text()
    fontsize = legend.prop.get_size_in_points()
    title_fontsize = legend.get_title().get_fontsize()
    frameon = legend.get_frame_on()
    legend.remove()
    grid._legend = grid.figure.legend(
        handles, labels, title=title, loc="center right", frameon=frameon,
        fontsize=fontsize, title_fontsize=title_fontsize,
    )


def mark_significant_heatmap_cells(axis, significant):
    significant = significant.fillna(False).astype(bool)
    for row_index, row in enumerate(significant.index):
        for column_index, column in enumerate(significant.columns):
            if significant.loc[row, column]:
                axis.scatter(
                    column_index + 0.82,
                    row_index + 0.18,
                    s=22,
                    marker="o",
                    facecolor="#111111",
                    edgecolor="white",
                    linewidth=0.7,
                    zorder=30,
                )


def add_significance_caption(figure, correction_scope="all displayed contrast cells"):
    figure.text(
        0.5,
        0.01,
        f"● Holm-adjusted p < {SIGNIFICANCE_ALPHA:.2f}; paired cluster-level sign-flip tests, corrected over {correction_scope}.",
        ha="center",
        va="bottom",
        fontsize=8,
    )


print(f"Loaded {len(episode_metrics):,} episode rows from {episode_metrics['source_csv'].nunique()} CSV files.")
display(
    episode_metrics.groupby(["model", "track", "condition_family", "time_mode"], dropna=False)
    .size()
    .rename("episodes")
    .reset_index()
    .sort_values(["track", "model", "condition_family", "time_mode"])
    .head(30)
)

import math

TRACK2_FAMILIES = ["2A.1", "2A.2", "2A.3", "2B.1", "2B.2"]
TRACK2_CONDITIONS = ["active", "full_context", "active_same_session_periodic_probe"]
TRACK2_FULL_CONTEXT_ABLATION_MODELS = ("Audio Flamingo Next", "Audio Flamingo Next Think")
TRACK2_RESULT_GLOB = "results_track2*"
TRACK2_ABSENCE_EVIDENCE_THRESHOLD = 0.80
TRACK2_DATASET_DIR = repo_root / "track2_dataset/generated_data/track2_role_ipc_test_v2/dataset"
T2_SCORE_COLUMNS = ["answer_score", "protocol_component_score", "exploration_component_score", "master_score", "primary_correct", "answer_complete", "acquisition_score", "present_acquisition", "nonredundant_query_rate", "redundant_query_rate", "belief_score", "calibration_score", "found_claim_support_score", "absence_factual_accuracy_score", "absence_claim_support_score", "undercovered_true_absence_claim_rate", "claim_support_score", "uncertainty_score", "budget_used", "found_claim_count", "absence_claim_count", "false_positive_control"]


def discover_track2_result_files(root):
    return frozen_input_paths(root, "track2_result_files")


def flatten_track2_result(path, root):
    result = json.loads(path.read_text(encoding="utf-8"))
    required = {"track2_result_schema_version", "task_id", "family", "variant", "condition", "run_status"}
    if not isinstance(result, dict) or not required <= set(result): return None
    relative = path.relative_to(root)
    row = {key: result.get(key) for key in ["task_id", "reel_id", "family", "variant", "condition", "run_status", "stop_reason", "final_answer_semantic_valid", "confidence"]}
    row.update(model_root=relative.parts[0], model=MODEL_NAME_OVERRIDES.get(relative.parts[0], relative.parts[0]), result_set=relative.parts[1], result_root=str(path.parents[3].relative_to(root)), source_json=str(relative), track=result["family"], episode_id=result["task_id"], condition_family=condition_family(result["condition"]))
    for group in ["answer_metrics", "evidence_metrics", "belief_metrics", "format_metrics"]:
        for key, value in (result.get(group) or {}).items():
            if not isinstance(value, (dict, list)): row[key] = value
    row.update(query_history=result.get("query_history", []), update_history=result.get("update_history", []), probe_history=result.get("probe_history", []), final_ledger=result.get("final_ledger", {}), format_records=result.get("format_records", []), target_evidence_coverage=(result.get("evidence_metrics") or {}).get("target_evidence_coverage", {}), absence_claim_coverage_history=(result.get("belief_metrics") or {}).get("absence_claim_coverage_history", []))
    return row


def load_track2_metrics(root):
    rows = [row for path in discover_track2_result_files(root) if (row := flatten_track2_result(path, root)) is not None]
    frame = pd.DataFrame(rows)
    if frame.empty: return pd.DataFrame(columns=["model_root", "model", "result_set", "condition", "family", "variant", "task_id", "reel_id", "run_status", "source_json"])
    key = ["model_root", "condition", "family", "variant", "task_id"]
    duplicate = frame.duplicated(key, keep=False)
    assert not duplicate.any(), "Duplicate Track 2 logical tasks:\n" + frame.loc[duplicate, key + ["result_set", "source_json"]].sort_values(key).to_string(index=False)
    return frame


def track2_weighted_mean(pairs):
    usable = [(float(value), float(weight)) for value, weight in pairs if value is not None and not isinstance(value, (dict, list)) and pd.notna(value)]
    return sum(value * weight for value, weight in usable) / sum(weight for _, weight in usable) if usable else np.nan


def track2_task_mode(row):
    key = f"{row.get('family')}/{row.get('variant')}"
    if key == "2A.1/present" or row.get("family") == "2A.2" or key == "2B.1/exact_count": return "present"
    if key == "2A.1/absent" or row.get("family") == "2A.3": return "absent"
    return "mixed"


def derive_track2_scores(frame):
    def derive(row):
        out = {name: np.nan for name in T2_SCORE_COLUMNS}
        if row.get("run_status") != "complete": return pd.Series(out)
        avg = lambda *names: track2_weighted_mean([(row.get(name), 1) for name in names])
        family, variant = row.get("family"), row.get("variant")
        primary, complete = row.get("primary_correct"), row.get("answer_complete")
        calibration = 1 - (float(row["confidence"]) - float(primary)) ** 2 if pd.notna(row.get("confidence")) and pd.notna(primary) else np.nan
        false_control = 1 - float(row["false_localization_rate"]) if pd.notna(row.get("false_localization_rate")) else np.nan
        if family == "2A.1" and variant == "present": core, structure, strict, localization = avg("present_absent_accuracy", "scene_identity_accuracy", "scene_rank_accuracy"), avg("scene_identity_accuracy", "scene_rank_accuracy"), primary, row.get("temporal_f1")
        elif family in {"2A.1", "2A.3"}: core, structure, strict, localization = track2_weighted_mean([(row.get("present_absent_accuracy"), 1), (false_control, 1)]), false_control, primary, np.nan
        elif family == "2A.2": core, structure, strict, localization = avg("localization_pass", "target_coverage", "predicted_interval_precision"), row.get("tolerant_temporal_iou"), primary, row.get("temporal_f1")
        elif family == "2B.1" and variant == "exact_count": core, structure, strict, localization = avg("exact_order_accuracy", "pairwise_order_accuracy", "card_to_scene_assignment_accuracy"), row.get("pairwise_order_accuracy"), primary, np.nan
        elif family == "2B.1": core, structure, strict, localization = avg("exact_order_accuracy", "pairwise_order_accuracy", "card_to_scene_assignment_accuracy", "absent_card_accuracy"), avg("pairwise_order_accuracy", "absent_card_accuracy"), primary, np.nan
        else: core, structure, strict, localization = track2_weighted_mean([(row.get(name), 1) for name in ["target_status_accuracy", "utterance_localization_accuracy", "scene_card_assignment_accuracy", "exact_order_accuracy", "pairwise_order_accuracy"]] + [(false_control, 1)]), avg("pairwise_order_accuracy", "scene_card_assignment_accuracy"), row.get("all_components_complete_accuracy"), row.get("mean_temporal_f1")
        answer = track2_weighted_mean([(core, .50), (structure, .20), (calibration, .10), (strict, .15), (localization, .05)])
        if pd.isna(complete) or float(complete) < 1 or not bool(row.get("final_answer_semantic_valid")): answer = 0.0
        mode = track2_task_mode(row)
        present = track2_weighted_mean([(row.get("mean_target_evidence_coverage"), .70), (row.get("all_present_targets_contacted"), .30)])
        absence = row.get("unique_reel_coverage")
        acquisition = present if mode == "present" else absence if mode == "absent" else track2_weighted_mean([(present, .70), (absence, .30)])
        total, replay = row.get("total_audio_inspected_seconds"), row.get("overlap_replay_seconds")
        if row.get("condition") == "full_context" or pd.isna(total): redundant, nonredundant = np.nan, np.nan
        elif float(total) == 0: redundant, nonredundant = np.nan, 0.0
        else: redundant = np.clip(float(replay) / float(total), 0, 1); nonredundant = 1 - redundant
        exploration = acquisition * nonredundant if pd.notna(acquisition) and pd.notna(nonredundant) else np.nan
        records = row.get("format_records") if isinstance(row.get("format_records"), list) else []; interaction = records[:-1]; protocol = np.nan
        if row.get("condition") != "full_context":
            format_valid = np.mean([bool(record.get("final_parse_valid")) for record in interaction]) if interaction else 0.0
            semantic_valid = np.mean([bool(record.get("semantic_valid", True)) for record in interaction]) if interaction else 0.0
            protocol = (0.0 if str(row.get("stop_reason", "")).startswith("invalid_action:") else 1.0) * math.sqrt(format_valid * semantic_valid)
        belief = track2_weighted_mean([(row.get("final_ledger_target_accuracy"), .40), (row.get("final_ledger_status_accuracy"), .25), (row.get("final_ledger_complete_accuracy"), .20), (row.get("final_ledger_answer_consistency"), .15)])
        updates = row.get("update_history") if isinstance(row.get("update_history"), list) else []
        found_count = sum(bool(record.get("valid")) and (record.get("update") or {}).get("status") == "FOUND" for record in updates)
        absence_history = row.get("absence_claim_coverage_history") if isinstance(row.get("absence_claim_coverage_history"), list) else []
        absence_history = [record for record in absence_history if isinstance(record, dict)]
        absence_count = len(absence_history)
        found_support = 1 - float(row.get("unsupported_found_claim_rate", 0)) if found_count else np.nan
        absence_factual_accuracy = np.nan
        absence_support = np.nan
        undercovered_true_absence = np.nan
        if absence_count:
            assert all("truth_is_absent" in record and pd.notna(record.get("unique_reel_coverage_fraction")) for record in absence_history), "Absence support requires truth and claim-time coverage"
            truth_absent = np.array([bool(record["truth_is_absent"]) for record in absence_history], dtype=bool)
            claim_coverage = np.array([float(record["unique_reel_coverage_fraction"]) for record in absence_history], dtype=float)
            assert np.isfinite(claim_coverage).all() and ((0 <= claim_coverage) & (claim_coverage <= 1)).all()
            supported_absence = truth_absent & (claim_coverage >= TRACK2_ABSENCE_EVIDENCE_THRESHOLD)
            undercovered_absence = truth_absent & ~supported_absence
            absence_factual_accuracy = float(truth_absent.mean())
            absence_support = float(supported_absence.mean())
            undercovered_true_absence = float(undercovered_absence.mean())
            raw_false_rate = float(row.get("false_absence_claim_rate", (~truth_absent).mean()))
            assert np.isclose(raw_false_rate + absence_support + undercovered_true_absence, 1.0), "Absence claim categories must partition all claims"
        claim_support = track2_weighted_mean([(found_support, found_count), (absence_support, absence_count)])
        uncertainty = track2_weighted_mean([(claim_support, 1), (calibration, 1)])
        master = answer if row.get("condition") == "full_context" else protocol ** .25 * exploration ** .35 * answer ** .40
        out.update(answer_score=answer, protocol_component_score=protocol, exploration_component_score=exploration, master_score=master, primary_correct=primary, answer_complete=complete, acquisition_score=acquisition, present_acquisition=present, nonredundant_query_rate=nonredundant, redundant_query_rate=redundant, belief_score=belief, calibration_score=calibration, found_claim_support_score=found_support, absence_factual_accuracy_score=absence_factual_accuracy, absence_claim_support_score=absence_support, undercovered_true_absence_claim_rate=undercovered_true_absence, claim_support_score=claim_support, uncertainty_score=uncertainty, budget_used=row.get("listening_budget_used_fraction"), found_claim_count=float(found_count), absence_claim_count=float(absence_count), false_positive_control=false_control)
        return pd.Series(out)
    result = frame.copy(); derived = result.apply(derive, axis=1) if not result.empty else pd.DataFrame(columns=T2_SCORE_COLUMNS, index=result.index)
    for column in T2_SCORE_COLUMNS: result[column] = derived[column]
    for column in [name for name in T2_SCORE_COLUMNS if name not in {"found_claim_count", "absence_claim_count"}]:
        finite = pd.to_numeric(result[column], errors="coerce").dropna(); assert finite.between(0, 1).all(), f"Track 2 {column} must be on the 0-1 scale"
    return result


def track2_slice(frame, conditions=None, families=None, variants=None, complete_only=True):
    out = frame.copy()
    if complete_only: out = out[out["run_status"] == "complete"]
    if conditions is not None: out = out[out["condition"].isin(conditions)]
    if families is not None: out = out[out["family"].isin(families)]
    if variants is not None: out = out[out["variant"].isin(variants)]
    return out


def track2_variant_summary(frame, value_cols, group_cols=("model", "condition", "family", "variant")):
    if frame.empty: return pd.DataFrame(columns=[*group_cols, *value_cols, "n_tasks", "n_reels"])
    grouped = frame.groupby(list(group_cols), dropna=False)
    return grouped[value_cols].mean().join(grouped.agg(n_tasks=("task_id", "nunique"), n_reels=("reel_id", "nunique"))).reset_index()


def track2_family_macro(frame, value_cols, group_cols=("model", "condition"), include_overall=False):
    if frame.empty: return pd.DataFrame(columns=[*group_cols, "family", *value_cols, "n_tasks", "n_reels"])
    variant = frame.groupby([*group_cols, "family", "variant"], dropna=False)[value_cols].mean().reset_index()
    family = variant.groupby([*group_cols, "family"], dropna=False)[value_cols].mean().reset_index()
    counts = frame.groupby([*group_cols, "family"], dropna=False).agg(n_tasks=("task_id", "nunique"), n_reels=("reel_id", "nunique")).reset_index(); family = family.merge(counts, on=[*group_cols, "family"])
    if include_overall:
        overall = family.groupby(list(group_cols), dropna=False)[value_cols].mean().reset_index(); overall["family"] = "Track 2 overall"
        total = frame.groupby(list(group_cols), dropna=False).agg(n_tasks=("task_id", "nunique"), n_reels=("reel_id", "nunique")).reset_index(); family = pd.concat([family, overall.merge(total, on=list(group_cols))], ignore_index=True)
    return family


def paired_track2_conditions(frame, value_cols):
    keys = ["model_root", "model", "family", "variant", "task_id", "reel_id"]
    left = track2_slice(frame, ["active"])[keys + value_cols]; right = track2_slice(frame, ["full_context"])[keys + value_cols]
    return left.merge(right, on=keys, suffixes=("_active", "_full_context"), validate="one_to_one")


def track2_component_long(frame, component_spec):
    records = []
    for family, columns in component_spec.items():
        for row in frame[frame["family"] == family].itertuples(index=False):
            for column in columns:
                value = getattr(row, column, np.nan)
                if pd.notna(value): records.append({"model": row.model, "family": family, "variant": row.variant, "task_id": row.task_id, "reel_id": row.reel_id, "component": column, "score": value})
    long = pd.DataFrame(records)
    if long.empty: return long
    variant = long.groupby(["model", "family", "variant", "component"], dropna=False)["score"].mean().reset_index(); family = variant.groupby(["model", "family", "component"], dropna=False)["score"].mean().reset_index()
    counts = long.groupby(["model", "family", "component"], dropna=False).agg(n_tasks=("task_id", "nunique"), n_reels=("reel_id", "nunique")).reset_index()
    return family.merge(counts, on=["model", "family", "component"])


track2_expected_tasks = pd.read_json(TRACK2_DATASET_DIR / "task_instances.jsonl", lines=True)
track2_manifest = json.loads((TRACK2_DATASET_DIR / "manifest.json").read_text())
track2_metrics = derive_track2_scores(load_track2_metrics(repo_root)); track2_complete = track2_slice(track2_metrics); track2_active = track2_slice(track2_metrics, ["active"])
_perfect = dict(run_status="complete", family="2A.1", variant="present", condition="active", primary_correct=1.0, answer_complete=1.0, final_answer_semantic_valid=True, confidence=1.0, present_absent_accuracy=1.0, scene_identity_accuracy=1.0, scene_rank_accuracy=1.0, temporal_f1=1.0, mean_target_evidence_coverage=1.0, all_present_targets_contacted=1.0, unique_reel_coverage=1.0, total_audio_inspected_seconds=1.0, overlap_replay_seconds=0.0, format_records=[{"final_parse_valid": True, "semantic_valid": True}, {"final_parse_valid": True, "semantic_valid": True}], update_history=[], absence_claim_coverage_history=[])
_checks = derive_track2_scores(pd.DataFrame([_perfect, {**_perfect, "condition": "full_context", "format_records": [_perfect["format_records"][-1]]}, {**_perfect, "stop_reason": "invalid_action:test"}, {**_perfect, "overlap_replay_seconds": .5}, {**_perfect, "family": "2A.1", "variant": "absent", "present_absent_accuracy": 1.0, "false_localization_rate": 0.0, "unique_reel_coverage": .8}, {**_perfect, "family": "2B.2", "variant": "structural_relational_retrieval", "target_status_accuracy": 1.0, "utterance_localization_accuracy": 1.0, "scene_card_assignment_accuracy": 1.0, "exact_order_accuracy": 1.0, "pairwise_order_accuracy": 1.0, "false_localization_rate": 0.0, "all_components_complete_accuracy": 1.0, "mean_temporal_f1": 1.0, "unique_reel_coverage": .2}, {"run_status": "api_error", "family": "2A.1", "variant": "present", "condition": "active"}, {**_perfect, "total_audio_inspected_seconds": 0.0, "overlap_replay_seconds": 0.0, "mean_target_evidence_coverage": 0.0, "all_present_targets_contacted": 0.0}]))
assert np.isclose(_checks.loc[0, ["protocol_component_score", "exploration_component_score", "answer_score", "master_score"]].astype(float), 1).all()
assert _checks.loc[1, "master_score"] == _checks.loc[1, "answer_score"] == 1 and _checks.loc[2, "protocol_component_score"] == _checks.loc[2, "master_score"] == 0
assert _checks.loc[3, "exploration_component_score"] < _checks.loc[0, "exploration_component_score"] and np.isclose(_checks.loc[4, "acquisition_score"], .8) and _checks.loc[5, "acquisition_score"] < 1 and pd.isna(_checks.loc[6, "master_score"])
assert _checks.loc[7, "nonredundant_query_rate"] == _checks.loc[7, "exploration_component_score"] == 0 and pd.isna(_checks.loc[1, "nonredundant_query_rate"])
_absence_claim_template = {"source": "final_answer", "target_id": "C_TARGET"}
_absence_checks = derive_track2_scores(pd.DataFrame([
    {**_perfect, "family": "2A.1", "variant": "absent", "false_absence_claim_rate": 0.0, "absence_claim_coverage_history": [{**_absence_claim_template, "truth_is_absent": True, "unique_reel_coverage_fraction": .90}]},
    {**_perfect, "family": "2A.1", "variant": "absent", "false_absence_claim_rate": 0.0, "absence_claim_coverage_history": [{**_absence_claim_template, "truth_is_absent": True, "unique_reel_coverage_fraction": .50}]},
    {**_perfect, "family": "2A.1", "variant": "absent", "false_absence_claim_rate": 1.0, "absence_claim_coverage_history": [{**_absence_claim_template, "truth_is_absent": False, "unique_reel_coverage_fraction": .90}]},
]))
assert np.allclose(_absence_checks["absence_factual_accuracy_score"], [1, 1, 0])
assert np.allclose(_absence_checks["absence_claim_support_score"], [1, 0, 0])
assert np.allclose(_absence_checks["undercovered_true_absence_claim_rate"], [0, 1, 0])
print(f"Loaded {len(track2_metrics):,} Track 2 result rows from {track2_metrics['source_json'].nunique() if not track2_metrics.empty else 0} JSON files.")
track2_main_inventory = track2_metrics[track2_metrics["condition"] != "full_context"]
if not track2_main_inventory.empty: display(track2_main_inventory.groupby(["model", "family", "condition", "run_status"], dropna=False).size().rename("results").reset_index())
else: display(Markdown("No Track 2 result JSON files were discovered."))


# v3 presentation aliases and evaluability fields.
episode_metrics["evidence_reach_score"] = episode_metrics["acquisition_score"]
episode_metrics["active_evidence_acquisition_score"] = episode_metrics["exploration_component_score"]
episode_metrics["temporal_belief_score"] = episode_metrics["belief_score"]
episode_metrics["budget_used"] = coalesce_columns(episode_metrics, ["budget_usage_rate", "step_usage_rate"])
episode_metrics["replay_rate"] = divide_or_nan(
    coalesce_columns(episode_metrics, ["repeated_bin_total", "repeated_seconds_total"]),
    coalesce_columns(episode_metrics, ["listened_bin_total", "listened_seconds_total"]),
).clip(0, 1)
episode_metrics["run_evaluable"] = episode_metrics["master_score"].notna()
target_recall = pd.to_numeric(episode_metrics.get("target_evidence_sentence_recall"), errors="coerce")
episode_metrics["target_evidence_found"] = target_recall.ge(1.0).astype("float64").where(target_recall.notna())
episode_metrics["stale_answer_rate_given_evidence"] = pd.to_numeric(episode_metrics.get("track1c_stale_answer_rate"), errors="coerce").where(episode_metrics["target_evidence_found"].eq(1))

track2_metrics["evidence_reach_score"] = track2_metrics["acquisition_score"]
track2_metrics["active_evidence_acquisition_score"] = track2_metrics["exploration_component_score"]
track2_metrics["temporal_belief_score"] = track2_metrics["belief_score"]
track2_metrics["replay_rate"] = track2_metrics["redundant_query_rate"]
track2_metrics["run_evaluable"] = track2_metrics["run_status"].eq("complete")
track2_metrics["strict_completion"] = (
    track2_metrics["run_status"].eq("complete")
    & track2_metrics["final_answer_semantic_valid"].fillna(False).astype(bool)
    & track2_metrics["primary_correct"].eq(1)
).astype(float)
track2_metrics["final_unresolved_ledger_rate"] = track2_metrics["final_ledger"].map(
    lambda ledger: (
        np.mean([entry.get("status") == "NOT_FOUND_YET" for entry in ledger.values()])
        if isinstance(ledger, dict) and ledger else np.nan
    )
)
track2_complete = track2_slice(track2_metrics)
track2_active = track2_slice(track2_metrics, ["active"])

track1_compatibility_frame = episode_metrics.assign(
    family=episode_metrics["track"],
    variant=episode_metrics["regime"],
    unit_id=episode_metrics["episode_id"],
)[[
    "model_root", "model", "track", "family", "variant", "unit_id", "condition_family",
    "master_score", "answer_score", "active_evidence_acquisition_score", "evidence_reach_score",
    "temporal_belief_score", "budget_used", "replay_rate", "run_evaluable",
]].copy()
track2_compatibility_frame = track2_metrics.assign(
    track="2", unit_id=track2_metrics["task_id"], condition_family=track2_metrics["condition"],
)[[
    "model_root", "model", "track", "family", "variant", "unit_id", "condition_family",
    "master_score", "answer_score", "active_evidence_acquisition_score", "evidence_reach_score",
    "temporal_belief_score", "budget_used", "replay_rate", "run_evaluable",
]].copy()
benchmark_metrics = pd.concat([track1_compatibility_frame, track2_compatibility_frame], ignore_index=True, sort=False)

SCOREBOARD_METRICS = ["master_score", "active_evidence_acquisition_score", "temporal_belief_score"]
SCOPE_ORDER = [
    "Overall Theory of Time",
    "Controlled Acoustic Temporal Mapping",
    "Controlled Conversational Belief Updating",
    "Long-Horizon Conversational Navigation",
]


def annotate_missing_heatmap_cells(axis, values, counts=None):
    for row_index, row in enumerate(values.index):
        for column_index, column in enumerate(values.columns):
            if pd.isna(values.loc[row, column]):
                count = 0 if counts is None or pd.isna(counts.loc[row, column]) else int(counts.loc[row, column])
                text = f"N/A\n(n={count})" if counts is not None else "N/A"
                axis.text(column_index + .5, row_index + .5, text, ha="center", va="center", fontsize=7, color="#555555")


def render_heatmap(table, title, *, cmap="YlGnBu", center=None, vmin=0, vmax=100, figsize=None, cbar_label="Score / difference (percentage points)", x_rotation=30, output_panel=None, output_filename=None):
    table = table.dropna(how="all")
    if table.empty:
        display(Markdown(f"No evaluable values are available for **{title}**."))
        return
    if figsize is None:
        figsize = (max(7, 1.3 * table.shape[1]), max(3.5, 0.32 * table.shape[0]))
    figure, axis = plt.subplots(figsize=figsize)
    sns.heatmap(table * 100, mask=table.isna(), annot=True, fmt=".1f", cmap=cmap, center=center, vmin=vmin, vmax=vmax, cbar_kws={"label": cbar_label}, ax=axis)
    axis.set_facecolor("#e6e6e6")
    annotate_missing_heatmap_cells(axis, table)
    plt.title(title)
    plt.xticks(rotation=x_rotation, ha="right" if x_rotation else "center")
    plt.yticks(rotation=0)
    plt.tight_layout()
    if output_panel is not None or output_filename is not None:
        assert output_panel is not None and output_filename is not None
        save_panel_figure(figure, output_panel, output_filename)
    plt.show()


def render_support_heatmap(values, counts, title, *, cmap="YlGnBu", center=None, vmin=0, vmax=100, figsize=None, x_rotation=None, show_counts=True, cbar_label="Score / difference (percentage points)"):
    values = values.copy().dropna(axis=0, how="all").dropna(axis=1, how="all")
    if values.empty:
        display(Markdown(f"No evaluable values are available for **{title}**."))
        return
    counts = counts.reindex(index=values.index, columns=values.columns).fillna(0)
    scaled = 100 * values
    annotations = pd.DataFrame("", index=values.index, columns=values.columns, dtype=object)
    for row in values.index:
        for column in values.columns:
            if pd.notna(values.loc[row, column]):
                annotations.loc[row, column] = f"{scaled.loc[row, column]:.1f}\n(n={int(counts.loc[row, column])})" if show_counts else f"{scaled.loc[row, column]:.1f}"
    if figsize is None:
        figsize = (max(7, 1.35 * values.shape[1]), max(4, 0.48 * values.shape[0]))
    plt.figure(figsize=figsize)
    ax = sns.heatmap(scaled, mask=values.isna(), annot=annotations, fmt="", cmap=cmap, center=center, vmin=vmin, vmax=vmax, cbar_kws={"label": cbar_label})
    ax.set_facecolor("#e6e6e6")
    for row_index, row in enumerate(values.index):
        for column_index, column in enumerate(values.columns):
            if pd.isna(values.loc[row, column]):
                ax.text(column_index + 0.5, row_index + 0.5, "N/A\n(n=0)" if show_counts else "N/A", ha="center", va="center", fontsize=8, color="#555555")
    if x_rotation is not None:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=x_rotation, ha="right" if x_rotation else "center")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def paired_track1_components(tracks, components):
    source = main_slice(episode_metrics, conditions=["active", "full_context"])
    source = source[source["track"].isin(tracks)].copy()
    source["paired_cluster_id"] = track1_paired_cluster_id(source)
    keys = ["model_root", "model", "track", "regime", "time_mode", "episode_id"]
    active = source[source["condition_family"] == "active"][keys + ["paired_cluster_id", *components]]
    full = source[source["condition_family"] == "full_context"][keys + components]
    return active.merge(full, on=keys, suffixes=("_active", "_full_context"), validate="one_to_one")


def component_gap_summary(tracks, component_labels):
    components = list(component_labels)
    paired = paired_track1_components(tracks, components)
    rows = []
    for (model, track), group in paired.groupby(["model", "track"], dropna=False):
        for component, label in component_labels.items():
            valid = group[[f"{component}_active", f"{component}_full_context", "paired_cluster_id"]].dropna()
            difference = valid[f"{component}_full_context"] - valid[f"{component}_active"]
            estimate, lower, upper, cluster_count, _ = clustered_mean_ci(difference, valid["paired_cluster_id"])
            p_value = cluster_sign_flip_p_value(difference, valid["paired_cluster_id"])
            rows.append({
                "model": model, "track": track, "component": label,
                "active_score": valid[f"{component}_active"].mean(),
                "full_context_score": valid[f"{component}_full_context"].mean(),
                "full_context_minus_active": estimate,
                "full_context_minus_active_ci_95_lower": lower,
                "full_context_minus_active_ci_95_upper": upper,
                "full_context_minus_active_p_value": p_value,
                "matched_episode_count": len(valid),
                "cluster_count": cluster_count,
            })
    return pd.DataFrame(rows)


def paired_prior_effects(tracks, metrics):
    source = episode_metrics[
        (episode_metrics["time_mode"] == MAIN_TIME_MODE)
        & (episode_metrics["condition_family"] == "active")
        & (episode_metrics["track"].isin(tracks))
    ].copy()
    source["paired_cluster_id"] = track1_paired_cluster_id(source)
    keys = ["model", "track", "episode_id"]
    rows = []
    for left, right, contrast in [
        ("Discovery", "Count-known", "Count-known - Discovery"),
        ("Count-known", "Inventory-known", "Inventory-known - Count-known"),
    ]:
        left_frame = source[source["regime"] == left][keys + ["paired_cluster_id", *metrics]]
        right_frame = source[source["regime"] == right][keys + metrics]
        paired = left_frame.merge(right_frame, on=keys, suffixes=("_left", "_right"), validate="one_to_one")
        for (model, track), group in paired.groupby(["model", "track"], dropna=False):
            for metric in metrics:
                valid = group[[f"{metric}_left", f"{metric}_right", "paired_cluster_id"]].dropna()
                difference = valid[f"{metric}_right"] - valid[f"{metric}_left"]
                estimate, lower, upper, cluster_count, _ = clustered_mean_ci(difference, valid["paired_cluster_id"])
                p_value = cluster_sign_flip_p_value(difference, valid["paired_cluster_id"])
                rows.append({
                    "model": model, "track": track, "contrast": contrast, "metric": metric,
                    "gain": estimate,
                    "gain_ci_95_lower": lower,
                    "gain_ci_95_upper": upper,
                    "gain_p_value": p_value,
                    "paired_episode_count": len(valid),
                    "cluster_count": cluster_count,
                })
    return pd.DataFrame(rows)


### Shared same-session trajectory asset

This is the v2 raw-result trajectory parser, retained for the Track 1C trajectory analysis in Panel 8 and the Track 1C stability/revision flag in Panel 15. Track 1A/1B probe trajectories are loaded for trace compatibility but are not interpreted as belief stability because evidence changes between probes.


In [ ]:
def discover_same_session_json_files(root):
    return frozen_input_paths(root, "same_session_result_files")


def probe_belief_score(metrics):
    values = [
        metrics.get("belief_bin_score_mean"),
        metrics.get("belief_pairwise_accuracy"),
        metrics.get("belief_localization_mean_iou"),
        metrics.get("belief_gap_accuracy"),
        metrics.get("belief_overlap_pair_f1"),
        metrics.get("belief_overlap_interval_iou"),
    ]
    values = [float(value) for value in values if pd.notna(value)]
    return float(np.mean(values)) if values else np.nan


def trajectory_rows_from_episode(path):
    episode = json.loads(path.read_text(encoding="utf-8"))
    model_root = path.relative_to(repo_root).parts[0]
    model = MODEL_NAME_OVERRIDES.get(model_root, model_root)
    track = normalize_track_id(episode.get("track_id"), path)
    time_mode = "continuous" if "results_continuous" in path.parts else "binned"
    base = {
        "model": model,
        "model_root": model_root,
        "time_mode": time_mode,
        "track": track,
        "episode_id": episode.get("episode_id"),
        "regime": episode.get("regime"),
        "trajectory_source_json": str(path.relative_to(repo_root)),
    }
    rows = []
    for probe_index, probe in enumerate(episode.get("probe_outputs", [])):
        score = probe_belief_score(probe.get("belief_metrics", {}))
        if pd.isna(score):
            continue
        rows.append({
            **base,
            "probe_index": probe_index,
            "checkpoint_step": probe.get("checkpoint_step"),
            "probe_kind": probe.get("probe_kind"),
            "trajectory_score": score,
            "trajectory_score_source": "probe_outputs.belief_metrics",
        })
    checkpoint_rows = []
    for probe_index, checkpoint in enumerate(episode.get("checkpoint_outputs", [])):
        score = checkpoint.get("score")
        if pd.isna(score):
            continue
        checkpoint_rows.append({
            **base,
            "probe_index": probe_index,
            "checkpoint_step": checkpoint.get("checkpoint_step"),
            "probe_kind": checkpoint.get("question_type", checkpoint.get("question_group", "checkpoint")),
            "trajectory_score": float(score),
            "trajectory_score_source": "checkpoint_outputs.score",
        })
    if checkpoint_rows:
        checkpoint_frame = pd.DataFrame(checkpoint_rows)
        grouped = (
            checkpoint_frame.groupby(["model", "model_root", "time_mode", "track", "episode_id", "regime", "trajectory_source_json", "checkpoint_step"], dropna=False)["trajectory_score"]
            .mean()
            .reset_index()
            .sort_values("checkpoint_step")
            .reset_index(drop=True)
        )
        grouped["probe_index"] = grouped.groupby(["model", "track", "episode_id"]).cumcount()
        grouped["probe_kind"] = "checkpoint_mean"
        grouped["trajectory_score_source"] = "checkpoint_outputs.score_mean_by_step"
        rows.extend(grouped.to_dict("records"))
    return rows



trajectory_rows = []
for episode_json in discover_same_session_json_files(repo_root):
    trajectory_rows.extend(trajectory_rows_from_episode(episode_json))
trajectory_metrics = pd.DataFrame(trajectory_rows)
trajectory_universe = main_slice(episode_metrics, conditions=["active_same_session"])[["model_root", "model", "track", "episode_id", "conversation_id"]].drop_duplicates()
if not trajectory_metrics.empty:
    source_key = ["model_root", "time_mode", "track", "regime", "episode_id"]
    sources = trajectory_metrics[source_key + ["trajectory_source_json"]].drop_duplicates()
    assert not sources.duplicated(source_key, keep=False).any(), "Multiple trajectory sources for one logical episode"
    trajectory_metrics = trajectory_metrics.sort_values(["model", "track", "episode_id", "checkpoint_step", "probe_index"])
    trajectory_episode_summary = trajectory_metrics.groupby(["model_root", "model", "track", "episode_id"], dropna=False).agg(
        first_probe_score=("trajectory_score", "first"),
        best_probe_score=("trajectory_score", "max"),
        final_probe_score=("trajectory_score", "last"),
        volatility=("trajectory_score", lambda values: values.diff().abs().mean()),
        n_probe_points=("trajectory_score", "count"),
        trajectory_source_json=("trajectory_source_json", "first"),
    ).reset_index()
else:
    trajectory_episode_summary = pd.DataFrame(columns=["model_root", "model", "track", "episode_id", "first_probe_score", "best_probe_score", "final_probe_score", "volatility", "n_probe_points", "trajectory_source_json"])
drift = trajectory_universe.merge(trajectory_episode_summary, on=["model_root", "model", "track", "episode_id"], how="left", validate="one_to_one")
drift["n_probe_points"] = drift["n_probe_points"].fillna(0).astype(int)
drift["drift"] = drift["best_probe_score"] - drift["final_probe_score"]
drift["trajectory_evaluable"] = drift["n_probe_points"] >= 2
drift.loc[~drift["trajectory_evaluable"], ["drift", "volatility"]] = np.nan
print(f"Loaded {len(trajectory_metrics):,} Track 1 trajectory points across {len(drift):,} episodes.")


## Section I. Overall Benchmark Performance

### Panel 1. Main Benchmark Scoreboard

The scoreboard uses ordinary active runs. Track 1A and 1B are equally weighted inside Controlled Acoustic Temporal Mapping; Track 1C defines Controlled Conversational Belief Updating; Track 2 is task-balanced within variant, variant-balanced within family, and equally weighted across its five families. Overall scores use 20/40/40 domain weights. A model is ranked only with 100% evaluable active-unit support across every required subtrack and family. Item-cluster bootstrap intervals and rank probabilities quantify sampling sensitivity of the reported ranks; they do not include model-generation stochasticity because repeated generations are unavailable.


In [ ]:
active_t1 = main_slice(episode_metrics, conditions=["active"]).copy()
expected_t1 = active_t1.groupby("track")["episode_id"].nunique().reindex(TRACKS).astype(int)
t1_means = active_t1[active_t1["run_evaluable"]].groupby(["model", "track"], dropna=False)[SCOREBOARD_METRICS].mean()
t1_support = active_t1.groupby(["model", "track"], dropna=False).agg(
    completed=("run_evaluable", "sum"), expected=("episode_id", "nunique")
).reset_index()

active_t2_all = track2_metrics[track2_metrics["condition"] == "active"].copy()
track2_unit_key = ["task_id", "family", "variant", "reel_id"]
assert not track2_expected_tasks.duplicated(track2_unit_key).any()
expected_t2_keys = set(map(tuple, track2_expected_tasks[track2_unit_key].itertuples(index=False, name=None)))
expected_t2 = len(expected_t2_keys)
t2_evaluable = active_t2_all[active_t2_all["run_evaluable"]]
t2_variant = t2_evaluable.groupby(["model", "family", "variant"], dropna=False)[SCOREBOARD_METRICS].mean().reset_index()
t2_family = t2_variant.groupby(["model", "family"], dropna=False)[SCOREBOARD_METRICS].mean().reset_index()
t2_domain = t2_family.groupby("model", dropna=False)[SCOREBOARD_METRICS].mean()
t2_support_rows = []
for model, model_rows in active_t2_all.groupby("model", dropna=False):
    observed_keys = set(map(tuple, model_rows[track2_unit_key].itertuples(index=False, name=None)))
    complete_keys = set(map(tuple, model_rows.loc[model_rows["run_evaluable"], track2_unit_key].itertuples(index=False, name=None)))
    t2_support_rows.append({
        "model": model, "completed": len(complete_keys & expected_t2_keys),
        "missing_expected": len(expected_t2_keys - complete_keys), "unexpected": len(observed_keys - expected_t2_keys),
        "represented_families": len({key[1] for key in complete_keys & expected_t2_keys}),
        "exact_expected_support": complete_keys == expected_t2_keys and observed_keys <= expected_t2_keys,
    })
t2_support = pd.DataFrame(t2_support_rows).set_index("model")

models = sorted(set(active_t1["model"]) | set(active_t2_all["model"]))
rows = []
for model in models:
    acoustic_parts = t1_means.loc[model].reindex(["1a", "1b"]) if model in t1_means.index.get_level_values(0) else pd.DataFrame(index=["1a", "1b"], columns=SCOREBOARD_METRICS)
    acoustic = acoustic_parts.mean().where(acoustic_parts.notna().sum() == 2)
    conversational = t1_means.loc[(model, "1c")] if (model, "1c") in t1_means.index else pd.Series(np.nan, index=SCOREBOARD_METRICS)
    long_horizon = t2_domain.loc[model] if model in t2_domain.index else pd.Series(np.nan, index=SCOREBOARD_METRICS)
    domain_frame = pd.DataFrame([acoustic, conversational, long_horizon], index=["acoustic", "conversational", "long_horizon"])
    domain_weights = pd.Series({"acoustic": 0.20, "conversational": 0.40, "long_horizon": 0.40})
    overall = domain_frame.mul(domain_weights, axis=0).sum().where(domain_frame.notna().all())

    support_rows = t1_support[t1_support["model"] == model].set_index("track")
    completed_1a = int(support_rows.loc["1a", "completed"]) if "1a" in support_rows.index else 0
    completed_1b = int(support_rows.loc["1b", "completed"]) if "1b" in support_rows.index else 0
    completed_1c = int(support_rows.loc["1c", "completed"]) if "1c" in support_rows.index else 0
    completed_2 = int(t2_support.loc[model, "completed"]) if model in t2_support.index else 0
    families_2 = int(t2_support.loc[model, "represented_families"]) if model in t2_support.index else 0
    missing_2 = int(t2_support.loc[model, "missing_expected"]) if model in t2_support.index else expected_t2
    unexpected_2 = int(t2_support.loc[model, "unexpected"]) if model in t2_support.index else 0
    exact_t2_support = bool(t2_support.loc[model, "exact_expected_support"]) if model in t2_support.index else False
    complete_support = (
        completed_1a == expected_t1["1a"] and completed_1b == expected_t1["1b"]
        and completed_1c == expected_t1["1c"] and exact_t2_support and families_2 == len(TRACK2_FAMILIES)
    )
    support_text = f"1A {completed_1a}/{expected_t1['1a']}; 1B {completed_1b}/{expected_t1['1b']}; 1C {completed_1c}/{expected_t1['1c']}; T2 {completed_2}/{expected_t2}, missing {missing_2}, unexpected {unexpected_2}, {families_2}/5 families"
    for scope, values in zip(SCOPE_ORDER, [overall, acoustic, conversational, long_horizon]):
        rows.append({"model": model, "scope": scope, **values.to_dict(), "completion_support": support_text, "rank_eligible": complete_support})

scoreboard = pd.DataFrame(rows)
overall_mask = scoreboard["scope"].eq("Overall Theory of Time") & scoreboard["rank_eligible"] & scoreboard["master_score"].notna()
scoreboard["rank"] = np.nan
scoreboard.loc[overall_mask, "rank"] = scoreboard.loc[overall_mask, "master_score"].rank(method="min", ascending=False)
assert scoreboard.loc[overall_mask, "rank"].notna().all()
assert scoreboard.loc[scoreboard["scope"].eq("Overall Theory of Time") & ~scoreboard["rank_eligible"], "rank"].isna().all()
scoreboard["rank_label"] = np.where(scoreboard["scope"].eq("Overall Theory of Time"), np.where(scoreboard["rank_eligible"], scoreboard["rank"].astype("Int64").astype(str), "PARTIAL"), "")
scoreboard["scope"] = pd.Categorical(scoreboard["scope"], SCOPE_ORDER, ordered=True)
scoreboard = scoreboard.sort_values(["model", "scope"]).reset_index(drop=True)
save_panel_table(scoreboard[["model", "scope", *SCOREBOARD_METRICS, "completion_support", "rank_label"]], "panel_01", "table_main_benchmark_scoreboard.csv")
display(percent_table(scoreboard[["model", "scope", *SCOREBOARD_METRICS, "completion_support", "rank_label"]], SCOREBOARD_METRICS))
score_heatmap = scoreboard.pivot(index="model", columns="scope", values="master_score").reindex(columns=SCOPE_ORDER)
score_heatmap.index = score_heatmap.index.map(short_model_label)
score_heatmap = score_heatmap.rename(columns={
    "Overall Theory of Time": "Overall", "Controlled Acoustic Temporal Mapping": "Acoustic\nmapping",
    "Controlled Conversational Belief Updating": "Conversational\nupdate",
    "Long-Horizon Conversational Navigation": "Long-horizon\nnavigation",
})
score_heatmap.index.name = "Model"; score_heatmap.columns.name = "Capability domain"
render_heatmap(score_heatmap, "Active master score by capability domain", figsize=(8.5, 4.8), cbar_label="Master score (%)", x_rotation=0, output_panel="panel_01", output_filename="plot_active_master_score_by_domain.png")

# Item-cluster bootstrap rank sensitivity. Common cluster draws preserve cross-model pairing.
rank_overall_mask = scoreboard["scope"].eq("Overall Theory of Time") & scoreboard["rank_eligible"] & scoreboard["master_score"].notna()
rank_models = scoreboard.loc[rank_overall_mask].sort_values("rank")["model"].tolist()

def model_cluster_matrix(frame, cluster_column):
    matrix = frame.groupby([cluster_column, "model"], dropna=False)["master_score"].mean().unstack("model").reindex(columns=rank_models)
    assert not matrix.empty and matrix.notna().all().all(), f"Complete paired support required for rank bootstrap: {cluster_column}"
    return matrix

rank_t1 = active_t1[active_t1["run_evaluable"]].copy()
rank_t1a = model_cluster_matrix(rank_t1[rank_t1["track"] == "1a"], "episode_id")
rank_t1b = model_cluster_matrix(rank_t1[rank_t1["track"] == "1b"], "episode_id")
rank_t1c = model_cluster_matrix(rank_t1[rank_t1["track"] == "1c"], "conversation_id")
t2_reel_variant = t2_evaluable.groupby(["model", "reel_id", "family", "variant"], dropna=False)["master_score"].mean().reset_index()
t2_reel_family = t2_reel_variant.groupby(["model", "reel_id", "family"], dropna=False)["master_score"].mean().reset_index()
assert t2_reel_family.groupby(["model", "reel_id"])["family"].nunique().eq(len(TRACK2_FAMILIES)).all()
t2_reel_domain = t2_reel_family.groupby(["model", "reel_id"], dropna=False)["master_score"].mean().reset_index()
rank_t2 = model_cluster_matrix(t2_reel_domain, "reel_id")

rank_point_scores = .20 * ((rank_t1a.mean().to_numpy() + rank_t1b.mean().to_numpy()) / 2) + .40 * rank_t1c.mean().to_numpy() + .40 * rank_t2.mean().to_numpy()
reported_point_scores = scoreboard.loc[rank_overall_mask].set_index("model").reindex(rank_models)["master_score"].to_numpy()
assert np.allclose(rank_point_scores, reported_point_scores)

RANK_BOOTSTRAP_REPLICATES = 10_000
RANK_BOOTSTRAP_RANDOM_SEED = 20260726
rank_rng = np.random.default_rng(RANK_BOOTSTRAP_RANDOM_SEED)

def paired_cluster_bootstrap_means(matrix):
    values = matrix.to_numpy(dtype=float)
    sampled = rank_rng.integers(0, len(values), size=(RANK_BOOTSTRAP_REPLICATES, len(values)))
    return values[sampled].mean(axis=1)

rank_bootstrap_scores = .20 * ((paired_cluster_bootstrap_means(rank_t1a) + paired_cluster_bootstrap_means(rank_t1b)) / 2) + .40 * paired_cluster_bootstrap_means(rank_t1c) + .40 * paired_cluster_bootstrap_means(rank_t2)
rank_bootstrap = pd.DataFrame(rank_bootstrap_scores, columns=rank_models).rank(axis=1, method="min", ascending=False)
rank_probability = pd.DataFrame({rank: rank_bootstrap.eq(rank).mean() for rank in range(1, len(rank_models) + 1)}, index=rank_models)
rank_uncertainty = pd.DataFrame({
    "model": rank_models,
    "point_score": rank_point_scores,
    "point_rank": scoreboard.loc[rank_overall_mask].set_index("model").reindex(rank_models)["rank"].to_numpy(),
    "bootstrap_score_ci_95_lower": np.quantile(rank_bootstrap_scores, .025, axis=0),
    "bootstrap_score_ci_95_upper": np.quantile(rank_bootstrap_scores, .975, axis=0),
    "median_bootstrap_rank": rank_bootstrap.median(axis=0).reindex(rank_models).to_numpy(),
    "bootstrap_rank_ci_95_lower": rank_bootstrap.quantile(.025, axis=0).reindex(rank_models).to_numpy(),
    "bootstrap_rank_ci_95_upper": rank_bootstrap.quantile(.975, axis=0).reindex(rank_models).to_numpy(),
    "probability_rank_1": rank_probability[1].to_numpy(),
    "probability_top_3": rank_bootstrap.le(3).mean(axis=0).reindex(rank_models).to_numpy(),
    "bootstrap_replicates": RANK_BOOTSTRAP_REPLICATES,
    "uncertainty_scope": "paired benchmark-item clusters only; excludes model-generation stochasticity",
})
for rank in rank_probability.columns:
    rank_uncertainty[f"probability_rank_{rank}"] = rank_probability[rank].to_numpy()
save_panel_table(rank_uncertainty, "panel_01", "table_scoreboard_rank_uncertainty.csv")
rank_probability_columns = [f"probability_rank_{rank}" for rank in rank_probability.columns]
display(Markdown("**Scoreboard rank sensitivity to paired benchmark-item resampling**")); display(percent_table(rank_uncertainty, ["point_score", "bootstrap_score_ci_95_lower", "bootstrap_score_ci_95_upper", "probability_rank_1", "probability_top_3", *rank_probability_columns]))

rank_probability_plot = rank_probability.copy(); rank_probability_plot.index = rank_probability_plot.index.map(short_model_label); rank_probability_plot.index.name = "Model"; rank_probability_plot.columns = [f"Rank {rank}" for rank in rank_probability_plot.columns]
figure, axis = plt.subplots(figsize=(10.5, 5.2))
sns.heatmap(100 * rank_probability_plot, annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100, cbar_kws={"label": "Bootstrap probability (%)"}, ax=axis)
axis.set(title="Overall rank sensitivity to paired benchmark-item resampling", xlabel="Rank", ylabel="Model")
figure.text(.5, .01, "Item-cluster uncertainty only; one model generation per task", ha="center", fontsize=8)
figure.tight_layout(rect=[0, .04, 1, 1])
save_panel_figure(figure, "panel_01", "plot_scoreboard_rank_uncertainty.png")
plt.show()


### Panel 2. Track 1 Active Performance Relative to Full Context

This matched Track 1 comparison uses answer scores, not condition-specific master scores, and pairs by model, track, regime, time mode, and episode ID. Paired slopes connect each model's full-context and active scores. For the compact plot only, Tracks 1A and 1B are combined as an equal-weight mean of their track-level scores; the displayed table retains the separate track results. Exact pairing does not make the scoring instruments identical: active behavioral questions are generated from explored events, full context marks all bins observed, and available answer-component weights can therefore differ. The gaps are descriptive differences under the two implemented condition-specific scoring contracts, not clean access-only causal effects; unasked active questions cannot be reconstructed post hoc.


In [ ]:
# Track 1 exact episode pairs.
t1_keys = ["model_root", "model", "track", "regime", "time_mode", "episode_id"]
t1_pair_source = main_slice(episode_metrics, conditions=["active", "full_context"])
t1_pair_source["paired_cluster_id"] = track1_paired_cluster_id(t1_pair_source)
t1_active_pair = t1_pair_source[t1_pair_source["condition_family"] == "active"][t1_keys + ["paired_cluster_id", "answer_score"]]
t1_full_pair = t1_pair_source[t1_pair_source["condition_family"] == "full_context"][t1_keys + ["answer_score"]]
t1_paired = t1_active_pair.merge(t1_full_pair, on=t1_keys, suffixes=("_active", "_full_context"), validate="one_to_one").dropna(subset=["answer_score_active", "answer_score_full_context"])
t1_gap_rows = []
for (model, track), group in t1_paired.groupby(["model", "track"], dropna=False):
    difference = group["answer_score_full_context"] - group["answer_score_active"]
    estimate, lower, upper, cluster_count, _ = clustered_mean_ci(difference, group["paired_cluster_id"])
    t1_gap_rows.append({
        "model": model, "track": track,
        "active_answer_score": group["answer_score_active"].mean(),
        "full_context_answer_score": group["answer_score_full_context"].mean(),
        "full_context_minus_active": estimate,
        "full_context_minus_active_ci_95_lower": lower,
        "full_context_minus_active_ci_95_upper": upper,
        "matched_unit_count": group["episode_id"].nunique(),
        "cluster_count": cluster_count,
    })
t1_gap = pd.DataFrame(t1_gap_rows)
t1_gap["scope"] = t1_gap["track"].str.upper()
answer_gap = t1_gap[["model", "scope", "active_answer_score", "full_context_answer_score", "full_context_minus_active", "full_context_minus_active_ci_95_lower", "full_context_minus_active_ci_95_upper", "matched_unit_count", "cluster_count"]].copy()
save_panel_table(answer_gap, "panel_02", "table_active_vs_full_context_answer_gap.csv")
display(percent_table(answer_gap, ["active_answer_score", "full_context_answer_score", "full_context_minus_active", "full_context_minus_active_ci_95_lower", "full_context_minus_active_ci_95_upper"]))
track1ab_source = answer_gap[answer_gap["scope"].isin(["1A", "1B"])]
assert track1ab_source.groupby("model")["scope"].nunique().eq(2).all(), "Track 1A/1B plot mean requires both track-level results for every model"
track1ab_mean = track1ab_source.groupby("model", as_index=False)[["active_answer_score", "full_context_answer_score"]].mean()
track1ab_mean["scope"] = "1A/1B mean"
plot_answer_gap = pd.concat([track1ab_mean, answer_gap[~answer_gap["scope"].isin(["1A", "1B"])]], ignore_index=True)
gap_model_order = sorted(answer_gap["model"].unique(), key=short_model_label)
gap_model_colors = model_palette(gap_model_order)
gap_panels = [(["1A/1B mean"], "Tracks 1A and 1B mean"), (["1C"], "Track 1C")]
figure, axes = plt.subplots(1, 2, figsize=(8.5, 4.8), sharey=True)
legend_handles = {}
for axis, (scopes, title) in zip(axes, gap_panels):
    positions = {scope: (2.4 * index, 2.4 * index + 1) for index, scope in enumerate(scopes)}
    for full_x, active_x in positions.values():
        axis.axvline(full_x, color="#9aa0a6", linestyle="--", linewidth=1, zorder=1)
        axis.axvline(active_x, color="#9aa0a6", linestyle="--", linewidth=1, zorder=1)
    for scope in scopes:
        scope_table = plot_answer_gap[plot_answer_gap["scope"] == scope].set_index("model").reindex(gap_model_order)
        full_x, active_x = positions[scope]
        for model in gap_model_order:
            full_value = scope_table.at[model, "full_context_answer_score"]
            active_value = scope_table.at[model, "active_answer_score"]
            if not (np.isfinite(full_value) and np.isfinite(active_value)):
                continue
            line, = axis.plot([full_x, active_x], [full_value, active_value], marker="o", markersize=5, linewidth=1.4, color=gap_model_colors[model], label=short_model_label(model), clip_on=False, zorder=10)
            legend_handles.setdefault(short_model_label(model), line)
    tick_positions = [position for scope in scopes for position in positions[scope]]
    tick_labels = ([label for scope in scopes for label in [f"{scope}\nFull context", f"{scope}\nActive"]] if len(scopes) > 1 else ["Full context", "Active"])
    axis.set_xticks(tick_positions, tick_labels)
    axis.set_xlim(-0.25, tick_positions[-1] + 0.25)
    axis.set_ylim(0, 1)
    axis.set_title(title)
    axis.set_xlabel("")
    axis.grid(axis="x", visible=False)
axes[0].set_ylabel("Matched answer score")
legend = figure.legend(legend_handles.values(), legend_handles.keys(), loc="lower center", bbox_to_anchor=(0.5, 0.055), ncol=5, frameon=False)
for handle in legend.get_lines():
    handle.set_linestyle("None")
figure.subplots_adjust(left=0.08, right=0.99, bottom=0.27, top=0.88, wspace=0.18)
save_panel_figure(figure, "panel_02", "plot_active_vs_full_context_slopes.png")
plt.show()


## Section II. Further Diagnostics

### A. Controlled Acoustic Temporal Mapping (Track 1A and Track 1B)

### Panel 3. Temporal Map Component Profile

This is the v2 component heatmap restricted to ordinary active, binned Discovery runs for Tracks 1A and 1B, reorganized with adjacent 1A/1B columns for each component. Event/bin map is whole-raster bin accuracy, so correctly predicted silent bins receive credit even when no event span is recovered; ordering instead scores every gold event pair and counts missing predicted spans as failures. An all-empty raster can therefore have a moderate event/bin-map score and zero ordering. The binned pipeline does not emit a separate overlap metric: Track 1A has no overlaps, while Track 1B overlap is represented in the multi-label raster but has no standalone score. The unavailable overlap column is therefore omitted rather than shown as missing or zero.


In [ ]:
component_labels_1ab = {
    "bin_map_score": "Event/bin map", "order_score": "Ordering", "localization_score": "Localization",
    "gap_score": "Gap reasoning",
}
profile_1ab = aggregate(main_slice(episode_metrics, conditions=["active"]), ["model", "track"], list(component_labels_1ab))
profile_1ab = profile_1ab[profile_1ab["track"].isin(["1a", "1b"])].copy()
save_panel_table(profile_1ab[["model", "track", *component_labels_1ab]], "panel_03", "table_temporal_map_component_profile.csv")
display(percent_table(profile_1ab[["model", "track", *component_labels_1ab]], list(component_labels_1ab)))
profile_column_order = [(metric, track) for metric in component_labels_1ab for track in ["1a", "1b"]]
profile_table = profile_1ab.pivot(index="model", columns="track", values=list(component_labels_1ab)).reindex(columns=pd.MultiIndex.from_tuples(profile_column_order))
profile_table.columns = [f"{component_labels_1ab[metric]}\n{track.upper()}" for metric, track in profile_column_order]
profile_table.index = profile_table.index.map(short_model_label); profile_table.index.name = "Model"
figure, axis = plt.subplots(figsize=(11.5, 5.2))
sns.heatmap(100 * profile_table, mask=profile_table.isna(), annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100, cbar_kws={"label": "Score (%)"}, ax=axis)
axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, profile_table)
for boundary in [2, 4, 6]: axis.axvline(boundary, color="white", linewidth=2.5)
axis.set(title="Active temporal-map component profile", xlabel="", ylabel="Model")
axis.set_xticklabels(["1A", "1B"] * len(component_labels_1ab), rotation=0)
for group_index, component in enumerate(component_labels_1ab.values()): axis.text(2 * group_index + 1, -.14, component, transform=axis.get_xaxis_transform(), ha="center", va="top", fontsize=9)
figure.subplots_adjust(left=.13, right=.94, bottom=.22, top=.91)
save_panel_figure(figure, "panel_03", "plot_temporal_map_component_profile.png")
plt.show()


<!-- Omitted analysis: Track 1A/1B probe-score changes conflate evidence acquisition with belief stability. -->


In [ ]:
# Analysis intentionally omitted; see the adjacent markdown rationale.


### Panel 4. Component-Level Active Versus Full-Context Gap

Exact matched episodes are compared for event/bin map, ordering, localization, and gaps, with adjacent 1A/1B columns in one aligned heatmap. Positive values mean full context outperforms active listening. For this panel only, the event/bin map is rescored from saved final rasters under a symmetric rule: unresolved `??` receives zero in both conditions, eliminating the active-only 0.5 credit for unobserved bins. Other component contracts can still differ because active questions depend on explored evidence, so the panel remains descriptive rather than an access-only causal estimate. A separate overlap gap is omitted because the selected binned pipeline has no standalone overlap score. With ten paired episodes and 80 Holm-corrected cells, the sign-flip family cannot attain adjusted p < 0.05; effect sizes and clustered intervals are reported, while absence of significance is not interpreted as absence of effects.


In [ ]:
binned_manifests = {
    track: json.loads((repo_root / f"generated_data/theory_of_time_v1/track_{track}/dataset/manifest.json").read_text())
    for track in ["1a", "1b"]
}
ground_truth_rasters = {
    (track, episode["episode_id"]): episode["ground_truth_raster"]
    for track, manifest in binned_manifests.items()
    for episode in manifest["episodes"]
}

def symmetric_final_raster_score(row):
    result_path = repo_root / row["result_root"] / "episode_results" / row["condition"] / row["regime"] / f"{row['episode_id']}.json"
    assert result_path.is_file(), f"Missing saved Track 1 result: {result_path}"
    result = json.loads(result_path.read_text())
    parsed = (result.get("final_raster") or {}).get("parsed") or {}
    gold = ground_truth_rasters[(row["track"], row["episode_id"])]
    predicted = parsed.get("bins")
    if not parsed.get("format_valid") or not isinstance(predicted, list):
        return 0.0
    assert len(predicted) == len(gold)
    return float(np.mean([candidate is not None and set(candidate) == set(target) for candidate, target in zip(predicted, gold)]))

symmetric_rows = main_slice(episode_metrics, conditions=["active", "full_context"])
symmetric_rows = symmetric_rows[symmetric_rows["track"].isin(["1a", "1b"])]
episode_metrics.loc[symmetric_rows.index, "symmetric_bin_map_score"] = symmetric_rows.apply(symmetric_final_raster_score, axis=1)
active_symmetric = episode_metrics.loc[symmetric_rows.index][episode_metrics.loc[symmetric_rows.index, "condition_family"].eq("active")]
full_symmetric = episode_metrics.loc[symmetric_rows.index][episode_metrics.loc[symmetric_rows.index, "condition_family"].eq("full_context")]
assert (active_symmetric["symmetric_bin_map_score"] <= active_symmetric["bin_map_score"] + 1e-12).all()
assert np.allclose(full_symmetric["symmetric_bin_map_score"], full_symmetric["bin_map_score"])

gap_component_labels_1ab = {
    "symmetric_bin_map_score": "Event/bin map", "order_score": "Ordering",
    "localization_score": "Localization", "gap_score": "Gap reasoning",
}
gap_1ab = component_gap_summary(["1a", "1b"], gap_component_labels_1ab)
gap_1ab["score_rule"] = np.where(gap_1ab["component"].eq("Event/bin map"), "symmetric unresolved=0 in both conditions", "existing component definition")
gap_1ab["full_context_minus_active_holm_p_value"] = holm_adjusted_p_values(gap_1ab["full_context_minus_active_p_value"])
gap_1ab["full_context_minus_active_significant"] = gap_1ab["full_context_minus_active_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
save_panel_table(gap_1ab, "panel_04", "table_component_active_vs_full_context_gap.csv")
display(percent_table(gap_1ab, ["active_score", "full_context_score", "full_context_minus_active", "full_context_minus_active_ci_95_lower", "full_context_minus_active_ci_95_upper"]))
gap_column_order = [(component, track) for component in gap_component_labels_1ab.values() for track in ["1a", "1b"]]
gap_table = gap_1ab.pivot(index="model", columns=["component", "track"], values="full_context_minus_active").reindex(columns=pd.MultiIndex.from_tuples(gap_column_order, names=["component", "track"]))
gap_significance = gap_1ab.pivot(index="model", columns=["component", "track"], values="full_context_minus_active_significant").reindex(index=gap_table.index, columns=gap_table.columns)
gap_table.columns = [f"{component}\n{track.upper()}" for component, track in gap_column_order]
gap_significance.columns = gap_table.columns
gap_table.index = gap_table.index.map(short_model_label); gap_table.index.name = "Model"
gap_significance.index = gap_table.index
vmax = max(0.05, np.nanmax(np.abs(gap_table.to_numpy()))) if gap_table.notna().any().any() else 0.05
figure, axis = plt.subplots(figsize=(11.5, 5.2))
sns.heatmap(100 * gap_table, mask=gap_table.isna(), annot=True, fmt=".1f", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, cbar_kws={"label": "Full context - active (pp)"}, ax=axis)
axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, gap_table)
mark_significant_heatmap_cells(axis, gap_significance)
for boundary in [2, 4, 6]: axis.axvline(boundary, color="white", linewidth=2.5)
axis.set(title="Track 1 component gap: matched full context - active", xlabel="", ylabel="Model")
axis.set_xticklabels(["1A", "1B"] * len(gap_component_labels_1ab), rotation=0)
for group_index, component in enumerate(gap_component_labels_1ab.values()): axis.text(2 * group_index + 1, -.14, component, transform=axis.get_xaxis_transform(), ha="center", va="top", fontsize=9)
add_significance_caption(figure)
figure.text(.5, .045, "Inference limitation: with n = 10 pairs and 80 Holm-corrected cells, adjusted p < 0.05 is unattainable.", ha="center", fontsize=8)
figure.subplots_adjust(left=.13, right=.94, bottom=.22, top=.91)
save_panel_figure(figure, "panel_04", "plot_component_active_vs_full_context_gap.png")
plt.show()


In [ ]:
display(Markdown("**Alternative line view — matched active and full-context component scores**"))
alternative_model_order = sorted(episode_metrics["model"].dropna().unique(), key=short_model_label)
alternative_model_colors = model_palette(alternative_model_order)
figure, axes = plt.subplots(2, len(gap_component_labels_1ab), figsize=(14, 6.8), sharex=True, sharey=True)
for row_index, track in enumerate(["1a", "1b"]):
    for column_index, component in enumerate(gap_component_labels_1ab.values()):
        axis = axes[row_index, column_index]
        block = gap_1ab[(gap_1ab["track"] == track) & (gap_1ab["component"] == component)]
        for row in block.itertuples(index=False):
            if pd.isna(row.active_score) or pd.isna(row.full_context_score): continue
            axis.plot([0, 1], 100 * np.array([row.active_score, row.full_context_score]), marker="o", markersize=4.5, linewidth=1.4, color=alternative_model_colors[row.model], label=short_model_label(row.model))
            if row.full_context_minus_active_significant:
                axis.scatter(1, 100 * row.full_context_score, s=32, facecolor="black", edgecolor="white", linewidth=.7, zorder=20)
        axis.set(title=component if row_index == 0 else "", ylabel=f"Track {track.upper()}\nScore (%)" if column_index == 0 else "")
        axis.set_xticks([0, 1], ["Active", "Full context"]); axis.set_ylim(-3, 103); axis.grid(axis="y", alpha=.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
figure.suptitle("Panel 4 alternative: matched component-score slopes", y=.99)
figure.text(.5, .93, "Inference limitation: n = 10 and the 80-cell Holm family make adjusted p < 0.05 unattainable; interpret slopes and intervals.", ha="center", fontsize=8.5)
figure.tight_layout(rect=[0, .11, 1, .88])
save_panel_figure(figure, "panel_04", "plot_component_score_slopes_alternative.png")
plt.show()


### Panel 5. Effect of Prior Information on Active Listening

Paired episode contrasts compare Count-known with Discovery and the implemented Inventory-known condition with Count-known. In Tracks 1A/1B, `labels_present` is supplied in chronological sequence, so the latter condition is relabelled **ordered-inventory-known** and estimates the bundled effect of inventory plus event order—not an inventory-only effect. Responses to a genuinely unordered inventory prompt do not exist and cannot be recovered without rerunning that condition. The compact grid uses contrasts as rows and tracks as columns. Support is reported separately for each metric; missing pairs are explicitly `N/A` and are not interpreted as paired effects. With ten paired episodes and roughly 100 cells per corrected contrast family, Holm-adjusted sign-flip inference is structurally uninformative; effect sizes and clustered intervals carry the interpretation.


In [ ]:
prior_metrics_1ab = ["master_score", "active_evidence_acquisition_score", "evidence_reach_score", "budget_used", "replay_rate"]
prior_1ab = paired_prior_effects(["1a", "1b"], prior_metrics_1ab)
prior_1ab["gain_holm_p_value"] = prior_1ab.groupby("contrast", dropna=False)["gain_p_value"].transform(holm_adjusted_p_values)
prior_1ab["gain_significant"] = prior_1ab["gain_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
prior_1ab["contrast_label"] = prior_1ab["contrast"].replace({"Inventory-known - Count-known": "Ordered-inventory-known - Count-known"})
save_panel_table(prior_1ab, "panel_05", "table_prior_information_effects.csv")
display(percent_table(prior_1ab, ["gain", "gain_ci_95_lower", "gain_ci_95_upper"]))
prior_metric_labels = {
    "master_score": "Master", "active_evidence_acquisition_score": "Acquisition",
    "evidence_reach_score": "Evidence reach", "budget_used": "Budget used", "replay_rate": "Replay rate",
}
prior_1ab_plot = prior_1ab.copy()
prior_1ab_plot["model_label"] = prior_1ab_plot["model"].map(short_model_label)
prior_1ab_plot["metric_label"] = prior_1ab_plot["metric"].map(prior_metric_labels)
vmax = max(0.05, np.nanmax(np.abs(prior_1ab_plot["gain"]))) if prior_1ab_plot["gain"].notna().any() else 0.05
contrast_order_1ab = [("Count-known - Discovery", "Count-known − Discovery"), ("Inventory-known - Count-known", "Ordered inventory-known − Count-known")]
prior_model_order = sorted(prior_1ab_plot["model_label"].unique())
figure, axes = plt.subplots(2, 2, figsize=(13.5, 8), sharex=True, sharey=True)
colorbar_axis = figure.add_axes([0.92, 0.16, 0.018, 0.68])
for row_index, (contrast, row_title) in enumerate(contrast_order_1ab):
    for column_index, track in enumerate(["1a", "1b"]):
        axis = axes[row_index, column_index]
        block = prior_1ab_plot[(prior_1ab_plot["contrast"] == contrast) & (prior_1ab_plot["track"] == track)]
        table = block.pivot(index="model_label", columns="metric_label", values="gain").reindex(index=prior_model_order, columns=list(prior_metric_labels.values()))
        significance = block.pivot(index="model_label", columns="metric_label", values="gain_significant").reindex(index=table.index, columns=table.columns)
        annotations = table.applymap(lambda value: f"{100 * value:.1f}" if pd.notna(value) else "")
        sns.heatmap(100 * table, mask=table.isna(), annot=annotations, fmt="", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, ax=axis, cbar=row_index == 0 and column_index == 0, cbar_ax=colorbar_axis if row_index == 0 and column_index == 0 else None)
        mark_significant_heatmap_cells(axis, significance)
        axis.set_facecolor("#e6e6e6")
        for y_index, model in enumerate(table.index):
            for x_index, metric in enumerate(table.columns):
                if pd.isna(table.loc[model, metric]): axis.text(x_index + .5, y_index + .5, "N/A", ha="center", va="center", fontsize=7, color="#555555")
        axis.set(title=f"Track {track.upper()}" if row_index == 0 else "", xlabel="Metric" if row_index == 1 else "", ylabel="")
        axis.tick_params(axis="x", rotation=0, labelsize=9, labelbottom=row_index == 1)
        axis.tick_params(axis="y", labelleft=column_index == 0)
figure.text(0.015, 0.70, contrast_order_1ab[0][1], rotation=90, va="center", ha="center", fontsize=11)
figure.text(0.015, 0.30, contrast_order_1ab[1][1], rotation=90, va="center", ha="center", fontsize=11)
colorbar_axis.set_ylabel("Prior effect (pp)")
figure.suptitle("Paired prior effects on active listening", y=0.98)
add_significance_caption(figure, "all displayed cells within each prior contrast")
figure.text(.5, .045, "Inference limitation: with n = 10 pairs and roughly 100 Holm-corrected cells per family, adjusted p < 0.05 is unattainable.", ha="center", fontsize=8)
figure.subplots_adjust(left=0.13, right=0.90, bottom=0.13, top=0.91, wspace=0.08, hspace=0.16)
save_panel_figure(figure, "panel_05", "plot_prior_information_effects.png")
plt.show()


In [ ]:
display(Markdown("**Alternative line view — complete three-regime paired trajectories**"))
prior_regime_order = ["Discovery", "Count-known", "Inventory-known"]
prior_line_source = episode_metrics[(episode_metrics["time_mode"] == MAIN_TIME_MODE) & (episode_metrics["condition_family"] == "active") & (episode_metrics["track"].isin(["1a", "1b"])) & (episode_metrics["regime"].isin(prior_regime_order))].copy()
prior_line_rows = []
for metric in prior_metrics_1ab:
    paired_values = prior_line_source.groupby(["model", "track", "episode_id", "regime"], dropna=False)[metric].mean().unstack("regime").reindex(columns=prior_regime_order).dropna()
    for (model, track), group in paired_values.groupby(level=["model", "track"], dropna=False):
        for regime in prior_regime_order:
            prior_line_rows.append({"model": model, "track": track, "metric": metric, "regime": regime, "score": group[regime].mean()})
prior_line = pd.DataFrame(prior_line_rows)
prior_significance_lookup = prior_1ab.set_index(["model", "track", "metric", "contrast"])["gain_significant"].to_dict()
figure, axes = plt.subplots(2, len(prior_metrics_1ab), figsize=(16, 6.8), sharex=True, sharey=True)
for row_index, track in enumerate(["1a", "1b"]):
    for column_index, metric in enumerate(prior_metrics_1ab):
        axis = axes[row_index, column_index]
        block = prior_line[(prior_line["track"] == track) & (prior_line["metric"] == metric)]
        for model in alternative_model_order:
            model_block = block[block["model"] == model].set_index("regime").reindex(prior_regime_order)
            if model_block["score"].isna().any(): continue
            scores = 100 * model_block["score"].to_numpy()
            axis.plot(range(3), scores, marker="o", markersize=4.5, linewidth=1.4, color=alternative_model_colors[model], label=short_model_label(model))
            for endpoint, contrast in [(1, "Count-known - Discovery"), (2, "Inventory-known - Count-known")]:
                if prior_significance_lookup.get((model, track, metric, contrast), False): axis.scatter(endpoint, scores[endpoint], s=32, facecolor="black", edgecolor="white", linewidth=.7, zorder=20)
        axis.set(title=prior_metric_labels[metric] if row_index == 0 else "", ylabel=f"Track {track.upper()}\nScore / rate (%)" if column_index == 0 else "")
        axis.set_xticks(range(3), ["Discovery", "Count\nknown", "Ordered inventory\nknown"]); axis.set_ylim(-3, 103); axis.grid(axis="y", alpha=.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
figure.suptitle("Panel 5 alternative: prior trajectories on complete three-regime pairs", y=.99)
figure.text(.5, .93, "Inference limitation: n = 10 and the large Holm family make adjusted p < 0.05 unattainable; interpret trajectories and intervals.", ha="center", fontsize=8.5)
figure.tight_layout(rect=[0, .11, 1, .88])
save_panel_figure(figure, "panel_05", "plot_prior_trajectories_alternative.png")
plt.show()


### B. Controlled Conversational Belief Updating (Track 1C)

### Panel 6. Semantic Temporal Belief Profile and Component-Level Active Versus Full-Context Gap

Aligned Blocks A and B share model rows. Block A profiles active semantic belief components; Block B reports matched full-context minus active gaps for semantic state, timing, speaker timeline, and final answer. Repeated nominal support counts are suppressed, while reduced support and `N/A (n=0)` remain explicit. Update success is active-only and is not compared with full context, which has no interactive update process. As in Panel 2, the active and full-context component opportunities and effective answer weights are condition-dependent, so Block B is a matched descriptive comparison rather than a clean access-only causal effect.


In [ ]:
component_labels_1c = {
    "semantic_score": "Semantic state", "localization_score": "Timing/localization",
    "speaker_score": "Speaker timeline", "update_score": "Update success", "answer_score": "Final answer",
}
active_1c_source = main_slice(episode_metrics, conditions=["active"])
active_1c_source = active_1c_source[active_1c_source["track"] == "1c"].copy()
active_1c = aggregate(active_1c_source, ["model", "track"], list(component_labels_1c)).set_index("model")
active_1c_support = active_1c_source.groupby("model")[list(component_labels_1c)].count()
profile_1c_table = active_1c[list(component_labels_1c)].rename(columns=component_labels_1c)
profile_1c_support = active_1c_support.rename(columns=component_labels_1c)
profile_1c_table.index = profile_1c_table.index.map(short_model_label); profile_1c_support.index = profile_1c_support.index.map(short_model_label)
save_panel_table(active_1c.reset_index(), "panel_06", "table_active_semantic_belief_profile.csv")
display(Markdown("**Block A — Active semantic temporal belief profile**")); display(percent_table(active_1c.reset_index(), list(component_labels_1c)))
gap_component_labels_1c = {metric: label for metric, label in component_labels_1c.items() if metric != "update_score"}
gap_1c = component_gap_summary(["1c"], gap_component_labels_1c)
gap_1c["full_context_minus_active_holm_p_value"] = holm_adjusted_p_values(gap_1c["full_context_minus_active_p_value"])
gap_1c["full_context_minus_active_significant"] = gap_1c["full_context_minus_active_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
save_panel_table(gap_1c, "panel_06", "table_component_active_vs_full_context_gap.csv")
display(Markdown("**Block B — Matched component-level full-context minus active gap**")); display(percent_table(gap_1c, ["active_score", "full_context_score", "full_context_minus_active", "full_context_minus_active_ci_95_lower", "full_context_minus_active_ci_95_upper"]))
gap_1c_table = gap_1c.pivot(index="model", columns="component", values="full_context_minus_active")
gap_1c_support = gap_1c.pivot(index="model", columns="component", values="matched_episode_count").reindex(index=gap_1c_table.index, columns=gap_1c_table.columns)
gap_1c_significance = gap_1c.pivot(index="model", columns="component", values="full_context_minus_active_significant").reindex(index=gap_1c_table.index, columns=gap_1c_table.columns)
gap_component_order_1c = ["Semantic state", "Timing/localization", "Speaker timeline", "Final answer"]
gap_1c_table = gap_1c_table.reindex(columns=gap_component_order_1c); gap_1c_support = gap_1c_support.reindex(columns=gap_component_order_1c); gap_1c_significance = gap_1c_significance.reindex(columns=gap_component_order_1c)
gap_1c_table.index = gap_1c_table.index.map(short_model_label); gap_1c_support.index = gap_1c_support.index.map(short_model_label)
gap_1c_significance.index = gap_1c_table.index
vmax = max(0.05, np.nanmax(np.abs(gap_1c_table.to_numpy()))) if not gap_1c_table.empty else 0.05

def selective_support_annotations(values, counts, decimals=1):
    annotations = pd.DataFrame("", index=values.index, columns=values.columns, dtype=object)
    nominal_support = counts.max(axis=0)
    for row in values.index:
        for column in values.columns:
            value = values.loc[row, column]
            count = int(counts.loc[row, column]) if pd.notna(counts.loc[row, column]) else 0
            if pd.notna(value):
                text = f"{100 * value:.{decimals}f}"
                if count < nominal_support.loc[column]: text += f"\n(n={count})"
                annotations.loc[row, column] = text
    return annotations


def annotate_missing_heatmap_cells(axis, values, counts=None):
    for row_index, row in enumerate(values.index):
        for column_index, column in enumerate(values.columns):
            if pd.isna(values.loc[row, column]):
                count = 0 if counts is None or pd.isna(counts.loc[row, column]) else int(counts.loc[row, column])
                axis.text(column_index + .5, row_index + .5, f"N/A\n(n={count})" if counts is not None else "N/A", ha="center", va="center", fontsize=7, color="#555555")


profile_label_map = {"Semantic state": "Semantic\nstate", "Timing/localization": "Timing /\nlocalization", "Speaker timeline": "Speaker\ntimeline", "Update success": "Update\nsuccess", "Final answer": "Final\nanswer"}
profile_1c_table = profile_1c_table.rename(columns=profile_label_map); profile_1c_support = profile_1c_support.rename(columns=profile_label_map)
gap_1c_table = gap_1c_table.rename(columns=profile_label_map); gap_1c_support = gap_1c_support.rename(columns=profile_label_map); gap_1c_significance = gap_1c_significance.rename(columns=profile_label_map)
model_order_1c = sorted(set(profile_1c_table.index) | set(gap_1c_table.index))
profile_1c_table = profile_1c_table.reindex(model_order_1c); profile_1c_support = profile_1c_support.reindex(model_order_1c).fillna(0)
gap_1c_table = gap_1c_table.reindex(model_order_1c); gap_1c_support = gap_1c_support.reindex(model_order_1c).fillna(0); gap_1c_significance = gap_1c_significance.reindex(model_order_1c).fillna(False)
figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [5, 4]})
for axis, values, counts, title, cmap, center, vmin, vmax_value, colorbar_label in [
    (axes[0], profile_1c_table, profile_1c_support, "A. Active component profile", "YlGnBu", None, 0, 100, "Score (%)"),
    (axes[1], gap_1c_table, gap_1c_support, "B. Matched full context - active", "vlag", 0, -100 * vmax, 100 * vmax, "Gap (pp)"),
]:
    annotations = selective_support_annotations(values, counts)
    sns.heatmap(100 * values, mask=values.isna(), annot=annotations, fmt="", cmap=cmap, center=center, vmin=vmin, vmax=vmax_value, cbar_kws={"label": colorbar_label, "shrink": .85}, ax=axis)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, values, counts)
    axis.set(title=title, xlabel="Component", ylabel="Model" if axis is axes[0] else "")
    axis.tick_params(axis="x", rotation=0, labelsize=9)
mark_significant_heatmap_cells(axes[1], gap_1c_significance)
axes[1].tick_params(axis="y", labelleft=False)
figure.suptitle("Track 1C semantic belief profile and matched component gap", y=0.99)
add_significance_caption(figure, "all displayed Block B contrast cells")
figure.tight_layout(rect=[0, .04, 1, .97])
save_panel_figure(figure, "panel_06", "plot_semantic_belief_profile_and_component_gap.png")
plt.show()


In [ ]:
display(Markdown("**Alternative line view — Block B matched active and full-context scores**"))
figure, axes = plt.subplots(1, len(gap_component_order_1c), figsize=(14, 4.8), sharex=True, sharey=True)
for axis, component in zip(axes, gap_component_order_1c):
    block = gap_1c[gap_1c["component"] == component]
    for row in block.itertuples(index=False):
        if pd.isna(row.active_score) or pd.isna(row.full_context_score): continue
        axis.plot([0, 1], 100 * np.array([row.active_score, row.full_context_score]), marker="o", markersize=4.5, linewidth=1.4, color=alternative_model_colors[row.model], label=short_model_label(row.model))
        if row.full_context_minus_active_significant:
            axis.scatter(1, 100 * row.full_context_score, s=32, facecolor="black", edgecolor="white", linewidth=.7, zorder=20)
    axis.set(title=component, xticks=[0, 1], xticklabels=["Active", "Full context"], ylim=(-3, 103)); axis.grid(axis="y", alpha=.2)
axes[0].set_ylabel("Score (%)")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
figure.suptitle("Panel 6 alternative: Track 1C matched component-score slopes", y=.99)
figure.text(.5, .92, "● Holm-adjusted p < 0.05 for the corresponding full-context minus active contrast", ha="center", fontsize=8.5)
figure.tight_layout(rect=[0, .14, 1, .86])
save_panel_figure(figure, "panel_06", "plot_component_score_slopes_alternative.png")
plt.show()


### Panel 7. Semantic Revision and Belief Inertia

The v2 revision computation is retained. Stale-answer claims are conditioned on all target evidence having been acquired and remain in the table and diagnosis. Scatter-marker area encodes the conditioned stale-answer rate; an `X` marks model-scenario cells where that rate is unavailable, distinguishing them from an evaluable rate of zero. Scatter facets use common `x = 0.5` and `y = 0.5` dashed references; these visual references are distinct from the diagnosis thresholds. The declared diagnosis is accompanied by lenient/declared/strict threshold sensitivity; these labels are descriptive conventions rather than validated clinical cut-points.


In [ ]:
revision_frame = main_slice(episode_metrics, conditions=["active"])
revision_frame = revision_frame[revision_frame["track"] == "1c"].copy()
if not revision_frame.empty:
    scenario_col = "category_label" if "category_label" in revision_frame.columns else "category" if "category" in revision_frame.columns else "regime"
    revision_frame["scenario"] = revision_frame[scenario_col].astype(str)
    revision_frame["target_evidence_found"] = pd.to_numeric(revision_frame.get("target_evidence_sentence_recall"), errors="coerce") >= 1.0
    base_revision = (
        revision_frame.groupby(["model", "scenario"], dropna=False)
        .agg(
            target_evidence_found=("target_evidence_found", "mean"),
            update_success=("track1c_update_success_rate", "mean"),
            active_answer_score=("answer_score", "mean"),
            n=("episode_id", "count"),
        )
        .reset_index()
    )
    found_only = revision_frame[revision_frame["target_evidence_found"]]
    stale_given_found = (
        found_only.groupby(["model", "scenario"], dropna=False)["track1c_stale_answer_rate"]
        .mean()
        .rename("stale_answer_rate_given_evidence")
        .reset_index()
    )
    revision = base_revision.merge(stale_given_found, on=["model", "scenario"], how="left")
    revision["diagnosis"] = np.select(
        [
            revision["target_evidence_found"] < EVIDENCE_THRESHOLD,
            revision["stale_answer_rate_given_evidence"] >= 0.30,
            revision["update_success"] >= LOW_SCORE_THRESHOLD,
        ],
        ["evidence missing", "belief inertia", "updates successfully"],
        default="mixed",
    )
    save_panel_table(revision, "panel_07", "table_semantic_revision_and_belief_inertia.csv")
    display(percent_table(revision, ["target_evidence_found", "update_success", "active_answer_score", "stale_answer_rate_given_evidence"]))

    revision_threshold_frames = []
    for threshold_scenario, thresholds in DIAGNOSTIC_THRESHOLD_SCENARIOS.items():
        scenario_revision = revision.drop(columns=["diagnosis"]).copy()
        scenario_revision["threshold_scenario"] = threshold_scenario
        scenario_revision["evidence_threshold"] = thresholds["evidence"]
        scenario_revision["stale_threshold"] = thresholds["stale"]
        scenario_revision["update_threshold"] = thresholds["low_score"]
        scenario_revision["diagnosis"] = np.select(
            [
                scenario_revision["target_evidence_found"] < thresholds["evidence"],
                scenario_revision["stale_answer_rate_given_evidence"] >= thresholds["stale"],
                scenario_revision["update_success"] >= thresholds["low_score"],
            ],
            ["evidence missing", "belief inertia", "updates successfully"],
            default="mixed",
        )
        revision_threshold_frames.append(scenario_revision)
    revision_threshold_sensitivity = pd.concat(revision_threshold_frames, ignore_index=True)
    save_panel_table(revision_threshold_sensitivity, "panel_07", "table_revision_threshold_sensitivity.csv")
    revision_threshold_summary = revision_threshold_sensitivity.groupby(["threshold_scenario", "diagnosis"], observed=False).size().rename("n_model_scenarios").reset_index()
    revision_threshold_summary["share"] = revision_threshold_summary["n_model_scenarios"] / revision_threshold_summary.groupby("threshold_scenario")["n_model_scenarios"].transform("sum")
    display(percent_table(revision_threshold_summary, ["share"]))

    diagnosis_order = ["evidence missing", "belief inertia", "mixed", "updates successfully"]
    revision_sensitivity_plot = revision_threshold_summary.pivot(index="threshold_scenario", columns="diagnosis", values="share").reindex(index=DIAGNOSTIC_THRESHOLD_SCENARIOS, columns=diagnosis_order, fill_value=0).fillna(0)
    axis = revision_sensitivity_plot.plot(kind="bar", stacked=True, figsize=(7.2, 4.2), color=sns.color_palette("Set2", len(diagnosis_order)))
    axis.set(title="Revision diagnosis sensitivity to threshold convention", xlabel="Threshold convention", ylabel="Share of model-scenario cells", ylim=(0, 1))
    axis.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); axis.tick_params(axis="x", rotation=0)
    axis.legend(title="Diagnosis", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    axis.figure.tight_layout()
    save_panel_figure(axis.figure, "panel_07", "plot_revision_threshold_sensitivity.png")
    plt.show()

    revision_plot = revision.dropna(subset=["target_evidence_found", "update_success"]).copy()
    if not revision_plot.empty:
        revision_plot["Model"] = revision_plot["model"].map(short_model_label)
        revision_plot["Stale answer rate"] = revision_plot["stale_answer_rate_given_evidence"].fillna(0.0).clip(0, 1)
        revision_plot["Stale-rate status"] = np.where(revision_plot["stale_answer_rate_given_evidence"].notna(), "Evaluable", "N/A")
        grid = sns.relplot(
            data=revision_plot,
            x="target_evidence_found",
            y="update_success",
            hue="Model",
            palette=model_palette(revision_plot["Model"].unique()),
            size="Stale answer rate",
            sizes=(45, 260),
            style="Stale-rate status",
            style_order=["Evaluable", "N/A"],
            markers={"Evaluable": "o", "N/A": "X"},
            col="scenario",
            col_wrap=3,
            height=3.0,
            aspect=1.0,
            alpha=0.85,
        )
        grid.set(xlim=(0, 1), ylim=(0, 1), xlabel="Target evidence found", ylabel="Update success")
        scenario_title_map = {
            "Cancellation and deletion": "Cancellation\nand deletion",
            "Disagreement and final resolution": "Disagreement and\nfinal resolution",
            "Temporal instruction sequence": "Temporal instruction\nsequence",
            "Correction and confirmation": "Correction and\nconfirmation",
            "Delayed answer and deferred resolution": "Delayed answer and\ndeferred resolution",
            "Topic switch with distractors": "Topic switch\nwith distractors",
        }
        grid.set_titles("")
        for ax, scenario in zip(grid.axes.flat, grid.col_names):
            ax.set_title(scenario_title_map.get(str(scenario), str(scenario)), fontsize=10)
            ax.set_xticks(np.linspace(0, 1, 6))
            ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f"))
            ax.tick_params(axis="x", labelsize=8.5)
            ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
            ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
            add_ideal_star(ax, 1.0, 1.0)
            for collection in ax.collections:
                collection.set_clip_on(False)
                collection.set_zorder(max(collection.get_zorder(), 10))
        add_ideal_star_legend(grid, "Ideal (1, 1)", next_section_label="Stale answer rate")
        if grid._legend is not None:
            grid._legend.set_frame_on(False)
            grid._legend.get_title().set_fontweight("bold")
            grid._legend.get_title().set_fontsize(11)
            for legend_text in grid._legend.texts:
                legend_section = legend_text.get_text()
                if legend_section in {"Model", "Stale answer rate", "Stale-rate status"}:
                    legend_text.set_fontweight("bold")
                    legend_text.set_fontsize(10.5)
                    if legend_section != "Model": legend_text.set_text(f"\n{legend_section}")
        grid.figure.subplots_adjust(right=0.79, wspace=0.12, hspace=0.36)
        save_panel_figure(grid.figure, "panel_07", "plot_semantic_revision_and_belief_inertia.png")
        plt.show()
else:
    display(Markdown("No Track 1c active rows are available in the selected slice."))


### Panel 8. Same-Session Semantic Belief Trajectory and Stability

This is a **probe-augmented semantic belief stability** diagnostic for Track 1C. It is distinct from Panel 7’s targeted revision analysis and is not interpreted as unprompted latent-memory degradation. To compare trajectories with different checkpoint counts, the plot retains episodes with at least two checkpoints and linearly interpolates each episode onto six equally spaced relative-progress positions; the first and final positions remain the observed episode endpoints.


In [ ]:
drift_1c = drift[drift["track"] == "1c"].copy()
if drift_1c.empty:
    display(Markdown("No Track 1C same-session trajectory data are available."))
else:
    summary_1c = drift_1c.groupby("model", dropna=False).agg(
        first_checkpoint_score=("first_probe_score", "mean"), best_checkpoint_score=("best_probe_score", "mean"),
        final_checkpoint_score=("final_probe_score", "mean"), best_minus_final_drift=("drift", "mean"),
        checkpoint_volatility=("volatility", "mean"), evaluable_checkpoint_points=("n_probe_points", "mean"),
        trajectory_evaluable_episode_rate=("trajectory_evaluable", "mean"), trajectory_episode_count=("episode_id", "nunique"),
    ).reset_index()
    drift_ci_rows = []
    for model, group in drift_1c.groupby("model", dropna=False):
        valid = group[["drift", "conversation_id"]].dropna()
        estimate, lower, upper, cluster_count, _ = clustered_mean_ci(valid["drift"], valid["conversation_id"])
        drift_ci_rows.append({
            "model": model,
            "best_minus_final_drift_ci_95_lower": lower,
            "best_minus_final_drift_ci_95_upper": upper,
            "best_minus_final_drift_cluster_count": cluster_count,
        })
        model_estimate = summary_1c.loc[summary_1c["model"] == model, "best_minus_final_drift"].iloc[0]
        assert (pd.isna(estimate) and pd.isna(model_estimate)) or np.isclose(estimate, model_estimate), "Panel 8 drift CI must preserve the existing point estimate"
    summary_1c = summary_1c.merge(pd.DataFrame(drift_ci_rows), on="model", validate="one_to_one")
    save_panel_table(summary_1c, "panel_08", "table_same_session_belief_stability.csv")
    display(percent_table(summary_1c, ["first_checkpoint_score", "best_checkpoint_score", "final_checkpoint_score", "best_minus_final_drift", "best_minus_final_drift_ci_95_lower", "best_minus_final_drift_ci_95_upper", "checkpoint_volatility", "trajectory_evaluable_episode_rate"]))
    trajectory_1c = trajectory_metrics[trajectory_metrics["track"] == "1c"].sort_values(["model", "episode_id", "checkpoint_step", "probe_index"])
    normalized_progress_grid = np.linspace(0.0, 1.0, 6)
    aligned_trajectory_rows = []
    for (model, episode_id), group in trajectory_1c.groupby(["model", "episode_id"], sort=False):
        episode_scores = group["trajectory_score"].to_numpy(dtype=float)
        if len(episode_scores) < 2:
            continue
        observed_progress = np.linspace(0.0, 1.0, len(episode_scores))
        aligned_scores = np.interp(normalized_progress_grid, observed_progress, episode_scores)
        for normalized_progress, trajectory_score in zip(normalized_progress_grid, aligned_scores):
            aligned_trajectory_rows.append({"model": model, "episode_id": episode_id, "normalized_progress": normalized_progress, "trajectory_score": trajectory_score})
    aligned_trajectory = pd.DataFrame(aligned_trajectory_rows)
    assert not aligned_trajectory.empty, "Panel 8 requires at least one Track 1C episode with two checkpoints"
    lines = aligned_trajectory.groupby(["model", "normalized_progress"], as_index=False)["trajectory_score"].mean()
    endpoint_check = lines[lines["normalized_progress"].isin([0.0, 1.0])].pivot(index="model", columns="normalized_progress", values="trajectory_score").reset_index()
    endpoint_check = endpoint_check.rename(columns={0.0: "aligned_first", 1.0: "aligned_final"})
    expected_endpoints = drift_1c[drift_1c["trajectory_evaluable"]].groupby("model", as_index=False).agg(expected_first=("first_probe_score", "mean"), expected_final=("final_probe_score", "mean"))
    endpoint_check = endpoint_check.merge(expected_endpoints, on="model", validate="one_to_one")
    assert np.allclose(endpoint_check["aligned_first"], endpoint_check["expected_first"]) and np.allclose(endpoint_check["aligned_final"], endpoint_check["expected_final"]), "Panel 8 aligned trajectory endpoints must match evaluable episode summaries"
    trajectory_models = sorted(lines["model"].unique(), key=short_model_label)
    trajectory_palette = model_palette(trajectory_models)
    figure, axis = plt.subplots(figsize=(11.8, 5.8))
    for model in trajectory_models:
        model_lines = lines[lines["model"] == model].sort_values("normalized_progress")
        axis.plot(100 * model_lines["normalized_progress"], 100 * model_lines["trajectory_score"], marker="o", linewidth=1.7, markersize=5.5, color=trajectory_palette[model])
    endpoints = lines[lines["normalized_progress"] == 1.0].sort_values("trajectory_score")
    label_positions = {}; previous = -np.inf
    for row in endpoints.itertuples(index=False):
        label_positions[row.model] = max(100 * float(row.trajectory_score), previous + 3.4)
        previous = label_positions[row.model]
    actual_upper = 100 * lines["trajectory_score"].max()
    overflow = max(label_positions.values(), default=actual_upper) - (actual_upper + 6.0)
    if overflow > 0: label_positions = {model: position - overflow for model, position in label_positions.items()}
    for row in endpoints.itertuples(index=False):
        endpoint_y = 100 * float(row.trajectory_score)
        label_y = label_positions[row.model]
        axis.plot([100, 102], [endpoint_y, label_y], color=trajectory_palette[row.model], linewidth=.9)
        axis.text(102.5, label_y, short_model_label(row.model), color=trajectory_palette[row.model], va="center", fontsize=8.5)
    plotted_scores = 100 * lines["trajectory_score"]
    axis.set(xlim=(-3, 112), ylim=(max(-2, plotted_scores.min() - 5), max(plotted_scores.max() + 5, max(label_positions.values()) + 3)), xlabel="Normalized progress through each episode’s observed checkpoint sequence", ylabel="Mean semantic belief score (%)", title="Episode-aligned Track 1C semantic belief trajectory")
    axis.set_xticks([0, 20, 40, 60, 80, 100], labels=["0%", "20%", "40%", "60%", "80%", "100%"])
    figure.tight_layout()
    save_panel_figure(figure, "panel_08", "plot_same_session_semantic_belief_trajectory.png")
    plt.show()


### Panel 9. Effect of Conversational Priors

Paired Track 1C episode contrasts include master, acquisition, target-evidence, update, inertia, and listening-cost measures.


In [ ]:
prior_metrics_1c = [
    "master_score", "active_evidence_acquisition_score", "evidence_reach_score",
    "target_evidence_found", "track1c_update_success_rate", "stale_answer_rate_given_evidence",
    "budget_used", "replay_rate",
]
prior_1c = paired_prior_effects(["1c"], prior_metrics_1c)
prior_1c["gain_holm_p_value"] = prior_1c.groupby("contrast", dropna=False)["gain_p_value"].transform(holm_adjusted_p_values)
prior_1c["gain_significant"] = prior_1c["gain_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
save_panel_table(prior_1c, "panel_09", "table_conversational_prior_effects.csv")
display(percent_table(prior_1c, ["gain", "gain_ci_95_lower", "gain_ci_95_upper"]))
prior_metric_labels_1c = {
    "master_score": "Master", "active_evidence_acquisition_score": "Acquisition",
    "evidence_reach_score": "Evidence reach", "target_evidence_found": "Evidence found",
    "track1c_update_success_rate": "Update success", "stale_answer_rate_given_evidence": "Stale answer",
    "budget_used": "Budget used", "replay_rate": "Replay rate",
}
prior_1c_plot = prior_1c.copy(); prior_1c_plot["model_label"] = prior_1c_plot["model"].map(short_model_label); prior_1c_plot["metric_label"] = prior_1c_plot["metric"].map(prior_metric_labels_1c)
vmax = max(0.05, np.nanmax(np.abs(prior_1c_plot["gain"]))) if prior_1c_plot["gain"].notna().any() else 0.05
contrast_order_1c = [("Count-known - Discovery", "Count-known − Discovery"), ("Inventory-known - Count-known", "Inventory-known − Count-known")]
prior_1c_model_order = sorted(prior_1c_plot["model_label"].unique())
figure, axes = plt.subplots(2, 1, figsize=(11.5, 8.5), sharex=True, sharey=True)
colorbar_axis = figure.add_axes([0.92, 0.16, 0.018, 0.68])
for index, (contrast, title) in enumerate(contrast_order_1c):
    axis = axes[index]
    contrast_data = prior_1c_plot[prior_1c_plot["contrast"] == contrast]
    table = contrast_data.pivot(index="model_label", columns="metric_label", values="gain").reindex(index=prior_1c_model_order, columns=list(prior_metric_labels_1c.values()))
    significance = contrast_data.pivot(index="model_label", columns="metric_label", values="gain_significant").reindex(index=table.index, columns=table.columns)
    annotations = table.applymap(lambda value: f"{100 * value:.1f}" if pd.notna(value) else "")
    sns.heatmap(100 * table, mask=table.isna(), annot=annotations, fmt="", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, ax=axis, cbar=index == 0, cbar_ax=colorbar_axis if index == 0 else None)
    mark_significant_heatmap_cells(axis, significance)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, table)
    axis.set(title=title, xlabel="Metric" if index == 1 else "", ylabel="Model")
    axis.tick_params(axis="x", rotation=0, labelsize=9, labelbottom=index == 1)
colorbar_axis.set_ylabel("Prior effect (pp)")
figure.suptitle("Conversational prior effects", y=0.98)
add_significance_caption(figure, "all displayed cells within each prior contrast")
figure.subplots_adjust(left=0.12, right=0.90, bottom=0.13, top=0.91, hspace=0.18)
save_panel_figure(figure, "panel_09", "plot_conversational_prior_effects.png")
plt.show()


### C. Long-Horizon Conversational Navigation (Track 2)

### Panel 10. Long-Horizon Active Task and Variant Profile

This panel reports the applicable active task components without converting non-applicable metrics to zero. Full-context Track 2 results are excluded from the main analysis.


In [ ]:
T2_COMPONENT_SPEC = {
    "2A.1": ["present_absent_accuracy", "scene_identity_accuracy", "scene_rank_accuracy", "temporal_f1"],
    "2A.2": ["localization_pass", "target_coverage", "predicted_interval_precision", "temporal_f1"],
    "2A.3": ["present_absent_accuracy", "false_positive_control"],
    "2B.1": ["exact_order_accuracy", "pairwise_order_accuracy", "card_to_scene_assignment_accuracy", "absent_card_accuracy"],
    "2B.2": ["target_status_accuracy", "utterance_localization_accuracy", "scene_card_assignment_accuracy", "pairwise_order_accuracy", "all_components_complete_accuracy"],
}
variant_order = [
    ("2A.1", "present"), ("2A.1", "absent"),
    ("2A.2", "exact_global"), ("2A.2", "exact_card_conditioned"), ("2A.2", "partial_global"), ("2A.2", "partial_card_conditioned"),
    ("2A.3", "absent_utterance"), ("2B.1", "exact_count"), ("2B.1", "extra_count"), ("2B.2", "structural_relational_retrieval"),
]
active_complete_t2 = track2_slice(track2_metrics, ["active"])
profile_rows = []
for (family, variant), group in active_complete_t2.groupby(["family", "variant"], dropna=False):
    applicable = T2_COMPONENT_SPEC[family]
    for model, model_group in group.groupby("model", dropna=False):
        record = {"model": model, "family": family, "variant": variant, "n_tasks": model_group["task_id"].nunique(), "n_reels": model_group["reel_id"].nunique()}
        for metric in applicable: record[metric] = pd.to_numeric(model_group.get(metric), errors="coerce").mean()
        record.update(strict_completion=model_group["strict_completion"].mean(), active_evidence_acquisition=model_group["active_evidence_acquisition_score"].mean(), listening_budget_used=model_group["budget_used"].mean())
        profile_rows.append(record)
t2_profile = pd.DataFrame(profile_rows)
t2_profile["cell"] = t2_profile["family"] + " / " + t2_profile["variant"]
track2_cell_order = [f"{family} / {variant}" for family, variant in variant_order]
t2_profile["cell"] = pd.Categorical(t2_profile["cell"], track2_cell_order, ordered=True)
save_panel_table(t2_profile.sort_values(["cell", "model"]), "panel_10", "table_active_task_and_variant_profile.csv")
display(Markdown("**Active task and variant profile**")); display(percent_table(t2_profile.sort_values(["cell", "model"]), [column for column in t2_profile if column not in {"model", "family", "variant", "cell", "n_tasks", "n_reels"}]))


### Ablation. Track 2 Full Context for Explicitly Supported Models

This ablation pairs exact active and full-context task IDs for **AFN** and **AFN-T** only. The table reports answer and strict-completion gaps by family and variant, and the heatmap summarizes the matched answer gaps without treating missing support as zero.


In [ ]:
track2_ablation_source = track2_metrics[track2_metrics["model"].isin(TRACK2_FULL_CONTEXT_ABLATION_MODELS)].copy()
paired_t2 = paired_track2_conditions(track2_ablation_source, ["answer_score", "strict_completion"])
assert set(paired_t2["model"].unique()) == set(TRACK2_FULL_CONTEXT_ABLATION_MODELS), "Track 2 full-context ablation requires both supported models"
ablation_key_columns = ["task_id", "family", "variant", "reel_id"]
expected_ablation_keys = set(map(tuple, track2_expected_tasks[ablation_key_columns].itertuples(index=False, name=None)))
for model, model_rows in paired_t2.groupby("model", dropna=False):
    observed_ablation_keys = set(map(tuple, model_rows[ablation_key_columns].itertuples(index=False, name=None)))
    assert observed_ablation_keys == expected_ablation_keys, f"{model} full-context ablation is missing {len(expected_ablation_keys - observed_ablation_keys)} expected pairs and has {len(observed_ablation_keys - expected_ablation_keys)} unexpected pairs"
if not paired_t2.empty:
    t2_condition_gap_rows = []
    for (model, family, variant), group in paired_t2.groupby(["model", "family", "variant"], dropna=False):
        answer_valid = group[["answer_score_active", "answer_score_full_context", "reel_id"]].dropna()
        answer_difference = answer_valid["answer_score_full_context"] - answer_valid["answer_score_active"]
        answer_gap, answer_lower, answer_upper, answer_cluster_count, _ = clustered_mean_ci(answer_difference, answer_valid["reel_id"])
        answer_p_value = cluster_sign_flip_p_value(answer_difference, answer_valid["reel_id"])
        strict_valid = group[["strict_completion_active", "strict_completion_full_context", "reel_id"]].dropna()
        strict_difference = strict_valid["strict_completion_full_context"] - strict_valid["strict_completion_active"]
        strict_gap, strict_lower, strict_upper, strict_cluster_count, _ = clustered_mean_ci(strict_difference, strict_valid["reel_id"])
        strict_p_value = cluster_sign_flip_p_value(strict_difference, strict_valid["reel_id"])
        t2_condition_gap_rows.append({
            "model": model, "family": family, "variant": variant,
            "active_answer_score": answer_valid["answer_score_active"].mean(),
            "full_context_answer_score": answer_valid["answer_score_full_context"].mean(),
            "answer_gap": answer_gap,
            "answer_gap_ci_95_lower": answer_lower,
            "answer_gap_ci_95_upper": answer_upper,
            "answer_gap_p_value": answer_p_value,
            "answer_gap_cluster_count": answer_cluster_count,
            "active_strict_completion": strict_valid["strict_completion_active"].mean(),
            "full_context_strict_completion": strict_valid["strict_completion_full_context"].mean(),
            "strict_completion_gap": strict_gap,
            "strict_completion_gap_ci_95_lower": strict_lower,
            "strict_completion_gap_ci_95_upper": strict_upper,
            "strict_completion_gap_p_value": strict_p_value,
            "strict_completion_gap_cluster_count": strict_cluster_count,
            "matched_task_count": group["task_id"].nunique(),
            "matched_reel_count": group["reel_id"].nunique(),
        })
    t2_condition_gap = pd.DataFrame(t2_condition_gap_rows)
    t2_condition_gap["answer_gap_holm_p_value"] = holm_adjusted_p_values(t2_condition_gap["answer_gap_p_value"])
    t2_condition_gap["answer_gap_significant"] = t2_condition_gap["answer_gap_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
    t2_condition_gap["strict_completion_gap_holm_p_value"] = holm_adjusted_p_values(t2_condition_gap["strict_completion_gap_p_value"])
    t2_condition_gap["strict_completion_gap_significant"] = t2_condition_gap["strict_completion_gap_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
    t2_condition_gap["cell"] = pd.Categorical(t2_condition_gap["family"] + " / " + t2_condition_gap["variant"], track2_cell_order, ordered=True)
    t2_condition_gap = t2_condition_gap.sort_values(["cell", "model"])
else:
    t2_condition_gap = pd.DataFrame()
save_panel_table(t2_condition_gap, "track2_full_context_ablation", "table_matched_active_vs_full_context_gap.csv")
display(Markdown("**Exact-task matched active versus full-context gap**")); display(percent_table(t2_condition_gap, ["active_answer_score", "full_context_answer_score", "answer_gap", "answer_gap_ci_95_lower", "answer_gap_ci_95_upper", "active_strict_completion", "full_context_strict_completion", "strict_completion_gap", "strict_completion_gap_ci_95_lower", "strict_completion_gap_ci_95_upper"]))
if not t2_condition_gap.empty:
    variant_short_labels = {
        "present": "Present", "absent": "Absent", "exact_global": "Exact\nglobal",
        "exact_card_conditioned": "Exact\ncard", "partial_global": "Partial\nglobal",
        "partial_card_conditioned": "Partial\ncard", "absent_utterance": "Absent\nutterance",
        "exact_count": "Exact\ncount", "extra_count": "Extra\ncount",
        "structural_relational_retrieval": "Relational",
    }
    vmax = max(.05, np.nanmax(np.abs(t2_condition_gap["answer_gap"])))
    gap_model_order = sorted(t2_condition_gap["model"].unique(), key=short_model_label)
    gap_table = t2_condition_gap.pivot(index="model", columns="cell", values="answer_gap").reindex(index=gap_model_order, columns=track2_cell_order)
    gap_significance = t2_condition_gap.pivot(index="model", columns="cell", values="answer_gap_significant").reindex(index=gap_table.index, columns=gap_table.columns)
    gap_table.index = gap_table.index.map(short_model_label)
    gap_significance.index = gap_table.index
    gap_table.columns = [f"{family}\n{variant_short_labels[variant]}" for family, variant in variant_order]
    gap_significance.columns = gap_table.columns
    figure, axis = plt.subplots(figsize=(14, 5.5))
    sns.heatmap(100 * gap_table, mask=gap_table.isna(), annot=True, fmt=".1f", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, cbar_kws={"label": "Full context - active (pp)"}, ax=axis)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, gap_table)
    mark_significant_heatmap_cells(axis, gap_significance)
    for boundary in [2, 6, 7, 9]: axis.axvline(boundary, color="white", linewidth=2.5)
    axis.set(title="Track 2 full-context ablation: matched answer gap by family and variant", xlabel="Family and variant", ylabel="Model")
    axis.tick_params(axis="x", rotation=0, labelsize=9)
    add_significance_caption(figure)
    figure.tight_layout(rect=[0, .04, 1, 1])
    save_panel_figure(figure, "track2_full_context_ablation", "plot_matched_answer_gap.png")
    plt.show()


### Panel 11. Navigation and Marginal Completion-Stage Profile

Two side-by-side heatmaps report independently evaluated evidence, contact, ledger, and answer-stage marginals by model: one is the equal-family mean over 2A.1/2A.2/2A.3 and the other is the equal-family mean over 2B.1/2B.2. The family-specific source table is retained. This is not a mathematical funnel: later rates may exceed earlier rates, and no causal stage dependence is implied. To reduce annotation density, zero-rate cells are intentionally unlabelled; nonzero values and unavailable cells remain explicit. The separate table below retains the stricter Track 2B.2 compound-task breakdown. Ordinary active runs only are used.


In [ ]:
stage_source = active_complete_t2.copy()
stage_source["relevant_evidence_reached"] = stage_source["evidence_reach_score"].ge(TRACK2_ABSENCE_EVIDENCE_THRESHOLD)
stage_source["required_targets_contacted"] = np.where(
    stage_source["all_present_targets_contacted"].notna(), stage_source["all_present_targets_contacted"].eq(1), stage_source["unique_reel_coverage"].ge(TRACK2_ABSENCE_EVIDENCE_THRESHOLD)
)
general_stages = [
    ("Relevant evidence reached", "relevant_evidence_reached"), ("Required targets contacted", "required_targets_contacted"),
    ("Final ledger target-correct", "final_ledger_target_accuracy"), ("Final ledger complete", "final_ledger_complete_accuracy"),
    ("Final answer task-core correct", "primary_correct"), ("Strict task complete", "strict_completion"),
]
rows = []
for row in stage_source.itertuples(index=False):
    for stage_index, (stage, metric) in enumerate(general_stages):
        value = getattr(row, metric, np.nan)
        rows.append({"model": row.model, "family": row.family, "task_id": row.task_id, "reel_id": row.reel_id, "stage": stage, "stage_index": stage_index, "complete": float(value == 1) if pd.notna(value) else np.nan})
general_stage_rows = pd.DataFrame(rows)
general_summary = general_stage_rows.groupby(["model", "family", "stage", "stage_index"], dropna=False).agg(marginal_completion_rate=("complete", "mean"), supported_tasks=("complete", "count"), n_reels=("reel_id", "nunique")).reset_index()
general_summary["rate_type"] = "independent marginal"
save_panel_table(general_summary, "panel_11", "table_marginal_completion_stage_profile.csv")
display(percent_table(general_summary, ["marginal_completion_rate"]))
stage_labels = ["Evidence\nreached", "Targets\ncontacted", "Ledger\ntarget", "Ledger\ncomplete", "Task\ncorrect", "Strict\ncomplete"]
stage_model_order = sorted(general_summary["model"].unique(), key=short_model_label)
general_plot_summary = general_summary.assign(task_group=general_summary["family"].str.slice(0, 2)).groupby(["model", "task_group", "stage", "stage_index"], dropna=False).agg(
    marginal_completion_rate=("marginal_completion_rate", "mean"), represented_families=("family", "nunique")
).reset_index()
general_plot_summary["rate_type"] = "independent marginal; equal-family mean within 2A or 2B"
save_panel_table(general_plot_summary, "panel_11", "table_marginal_completion_stage_profile_grouped.csv")
figure, axes = plt.subplots(1, 2, figsize=(12.8, 5.3), sharey=True)
colorbar_axis = figure.add_axes([0.92, 0.19, 0.018, 0.62])
for index, (task_group, title) in enumerate([("2A", "(a) Track 2A"), ("2B", "(b) Track 2B")]):
    axis = axes[index]
    group_table = general_plot_summary[general_plot_summary["task_group"] == task_group].pivot(index="model", columns="stage_index", values="marginal_completion_rate").reindex(index=stage_model_order, columns=range(len(general_stages)))
    group_table.index = group_table.index.map(short_model_label); group_table.columns = stage_labels
    group_annotations = group_table.applymap(lambda value: f"{100 * value:.0f}" if pd.notna(value) and value > 0 else "")
    sns.heatmap(100 * group_table, mask=group_table.isna(), annot=group_annotations, fmt="", annot_kws={"fontsize": 8}, cmap="YlGnBu", vmin=0, vmax=100, ax=axis, cbar=index == 0, cbar_ax=colorbar_axis if index == 0 else None)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, group_table)
    axis.set(title=f"{title} (family-balanced)", xlabel="", ylabel="")
    axis.tick_params(axis="x", rotation=15, labelsize=9, pad=3)
    # Long stage names need a smaller increase to fit the original column widths.
    for label in axis.get_xticklabels():
        label._v5_font_scale = 1.02
    if index == 1: axis.tick_params(axis="y", labelleft=False)
colorbar_axis.set_ylabel("Completion rate (%)")
figure.subplots_adjust(left=0.09, right=0.90, bottom=0.14, top=0.82, wspace=0.10)
save_panel_figure(figure, "panel_11", "plot_marginal_completion_stage_profile.png")
plt.show()

specific_2b2 = [
    ("All three present utterances contacted", "all_present_targets_contacted"), ("All target statuses correct", "target_status_accuracy"),
    ("All present utterances localized", "utterance_localization_accuracy"), ("All scene assignments correct", "scene_card_assignment_accuracy"),
    ("Present utterances ordered correctly", "exact_order_accuracy"), ("All components complete", "all_components_complete_accuracy"),
]
source_2b2 = stage_source[stage_source["family"] == "2B.2"]
rows_2b2 = [{"model": row.model, "task_id": row.task_id, "reel_id": row.reel_id, "stage": label, "stage_index": index, "complete": float(getattr(row, metric) == 1) if pd.notna(getattr(row, metric, np.nan)) else np.nan} for row in source_2b2.itertuples(index=False) for index, (label, metric) in enumerate(specific_2b2)]
summary_2b2 = pd.DataFrame(rows_2b2).groupby(["model", "stage", "stage_index"], dropna=False).agg(completion_rate=("complete", "mean"), supported_tasks=("complete", "count"), n_reels=("reel_id", "nunique")).reset_index()
save_panel_table(summary_2b2, "panel_11", "table_track2b2_compound_completion.csv")
display(percent_table(summary_2b2, ["completion_rate"]))
cost_summary = stage_source.groupby(["model", "family"], dropna=False).agg(
    cost_to_contact_all_present_targets=("cost_to_all_present_target_contact_seconds", "mean"),
    cost_to_all_correct_commitments=("cost_to_all_correct_commitments_seconds", "mean"),
    final_ledger_answer_consistency=("final_ledger_answer_consistency", "mean"), n_tasks=("task_id", "nunique"), n_reels=("reel_id", "nunique"),
).reset_index()
save_panel_table(cost_summary, "panel_11", "table_navigation_cost_and_consistency.csv")
display(percent_table(cost_summary, ["final_ledger_answer_consistency"]))


### Panel 12. Exact/Partial Quotation and Search-Scope Effects

The Track 2A.2 two-by-two design is summarized from ordinary active runs as within-reel balanced cell contrasts. These are not treated as guaranteed same-target pairs. One aligned heatmap combines active outcome and target-contact contrasts, and each column explicitly names its contrast. Contact cost remains a separate table because it is conditional on actually contacting a target.


In [ ]:
quote_source = track2_slice(track2_metrics, conditions=["active"], families=["2A.2"]).copy()
variant_map = {
    "exact_global": ("exact", "global"), "exact_card_conditioned": ("exact", "card-conditioned"),
    "partial_global": ("partial", "global"), "partial_card_conditioned": ("partial", "card-conditioned"),
}
quote_source[["quote_form", "search_scope"]] = quote_source["variant"].map(variant_map).apply(pd.Series)
quote_metrics = ["localization_pass", "temporal_f1", "target_contact_rate", "mean_cost_to_first_target_contact_seconds", "answer_score"]
cell_means = quote_source.groupby(["model", "condition", "reel_id", "quote_form", "search_scope"], dropna=False)[quote_metrics].mean().reset_index()
contrast_rows = []
for (model, condition), group in cell_means.groupby(["model", "condition"], dropna=False):
    for metric in quote_metrics:
        table = group.pivot_table(index="reel_id", columns=["quote_form", "search_scope"], values=metric)
        partial_terms = []
        conditioning_terms = []
        for scope in ["global", "card-conditioned"]:
            if ("exact", scope) in table and ("partial", scope) in table: partial_terms.append(table[("exact", scope)] - table[("partial", scope)])
        for form in ["exact", "partial"]:
            if (form, "card-conditioned") in table and (form, "global") in table: conditioning_terms.append(table[(form, "card-conditioned")] - table[(form, "global")])
        partial_values = pd.concat(partial_terms, axis=1).mean(axis=1).dropna() if partial_terms else pd.Series(dtype=float)
        conditioning_values = pd.concat(conditioning_terms, axis=1).mean(axis=1).dropna() if conditioning_terms else pd.Series(dtype=float)
        partial_clusters = pd.Series(partial_values.index, index=partial_values.index, dtype="string")
        conditioning_clusters = pd.Series(conditioning_values.index, index=conditioning_values.index, dtype="string")
        partial_estimate, partial_lower, partial_upper, partial_cluster_count, _ = clustered_mean_ci(partial_values, partial_clusters)
        conditioning_estimate, conditioning_lower, conditioning_upper, conditioning_cluster_count, _ = clustered_mean_ci(conditioning_values, conditioning_clusters)
        partial_p_value = cluster_sign_flip_p_value(partial_values, partial_clusters)
        conditioning_p_value = cluster_sign_flip_p_value(conditioning_values, conditioning_clusters)
        contrast_rows.append({
            "model": model, "condition": condition, "metric": metric,
            "partial_quote_penalty": partial_estimate,
            "partial_quote_penalty_ci_95_lower": partial_lower,
            "partial_quote_penalty_ci_95_upper": partial_upper,
            "partial_quote_penalty_p_value": partial_p_value,
            "card_conditioning_gain": conditioning_estimate,
            "card_conditioning_gain_ci_95_lower": conditioning_lower,
            "card_conditioning_gain_ci_95_upper": conditioning_upper,
            "card_conditioning_gain_p_value": conditioning_p_value,
            "supported_reels_partial": partial_cluster_count,
            "supported_reels_conditioning": conditioning_cluster_count,
        })
quote_contrasts = pd.DataFrame(contrast_rows)
score_contrast = quote_contrasts["metric"] != "mean_cost_to_first_target_contact_seconds"
quote_contrasts["partial_quote_penalty_holm_p_value"] = np.nan
quote_contrasts.loc[score_contrast, "partial_quote_penalty_holm_p_value"] = holm_adjusted_p_values(quote_contrasts.loc[score_contrast, "partial_quote_penalty_p_value"])
quote_contrasts["partial_quote_penalty_significant"] = quote_contrasts["partial_quote_penalty_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
quote_contrasts["card_conditioning_gain_holm_p_value"] = np.nan
quote_contrasts.loc[score_contrast, "card_conditioning_gain_holm_p_value"] = holm_adjusted_p_values(quote_contrasts.loc[score_contrast, "card_conditioning_gain_p_value"])
quote_contrasts["card_conditioning_gain_significant"] = quote_contrasts["card_conditioning_gain_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
save_panel_table(quote_contrasts, "panel_12", "table_quotation_and_search_scope_contrasts.csv")
quote_contrasts_display = quote_contrasts.copy()
score_contrast = quote_contrasts_display["metric"] != "mean_cost_to_first_target_contact_seconds"
for column in ["partial_quote_penalty", "partial_quote_penalty_ci_95_lower", "partial_quote_penalty_ci_95_upper", "card_conditioning_gain", "card_conditioning_gain_ci_95_lower", "card_conditioning_gain_ci_95_upper"]:
    quote_contrasts_display.loc[score_contrast, column] = (100 * quote_contrasts_display.loc[score_contrast, column]).round(1)
    quote_contrasts_display.loc[~score_contrast, column] = quote_contrasts_display.loc[~score_contrast, column].round(1)
display(quote_contrasts_display)
outcome_metrics = {"answer_score": "Answer", "localization_pass": "Localization pass", "temporal_f1": "Temporal F1"}
outcome_source = quote_contrasts[(quote_contrasts["condition"] == "active") & quote_contrasts["metric"].isin(outcome_metrics)]
outcome_table = outcome_source.pivot(index="model", columns="metric", values="card_conditioning_gain").rename(columns=outcome_metrics)
outcome_significance = outcome_source.pivot(index="model", columns="metric", values="card_conditioning_gain_significant").rename(columns=outcome_metrics)
outcome_table = outcome_table.dropna(axis=0, how="all"); outcome_significance = outcome_significance.reindex(index=outcome_table.index, columns=outcome_table.columns)
outcome_table.index = outcome_table.index.map(short_model_label); outcome_significance.index = outcome_table.index
active_search = quote_contrasts[(quote_contrasts["condition"] == "active") & quote_contrasts["metric"].isin(["target_contact_rate", "mean_cost_to_first_target_contact_seconds"])].copy()
contact_gain = active_search[active_search["metric"] == "target_contact_rate"].set_index("model")[["card_conditioning_gain"]].rename(columns={"card_conditioning_gain": "Target contact"})
contact_significance = active_search[active_search["metric"] == "target_contact_rate"].set_index("model")[["card_conditioning_gain_significant"]].rename(columns={"card_conditioning_gain_significant": "Target contact"})
contact_support = active_search[active_search["metric"] == "target_contact_rate"].set_index("model")[["supported_reels_conditioning"]].rename(columns={"supported_reels_conditioning": "Target contact"})
contact_gain.index = contact_gain.index.map(short_model_label); contact_support.index = contact_support.index.map(short_model_label); contact_significance.index = contact_gain.index
conditioning_table = outcome_table.reindex(columns=["Answer", "Localization pass", "Temporal F1"]).join(contact_gain, how="outer")
conditioning_significance = outcome_significance.reindex(columns=["Answer", "Localization pass", "Temporal F1"]).join(contact_significance, how="outer").reindex(index=conditioning_table.index, columns=conditioning_table.columns)
conditioning_table.columns = ["Answer\n(active)", "Localization pass\n(active)", "Temporal F1\n(active)", "Target contact\n(active)"]
conditioning_significance.columns = conditioning_table.columns
conditioning_table.index.name = "Model"; conditioning_table.columns.name = "Metric"
if not conditioning_table.empty:
    vmax = max(.05, np.nanmax(np.abs(conditioning_table.to_numpy())))
    annotations = conditioning_table.applymap(lambda value: f"{100 * value:.1f}" if pd.notna(value) else "")
    figure, axis = plt.subplots(figsize=(8.5, 5.2))
    sns.heatmap(100 * conditioning_table, mask=conditioning_table.isna(), annot=annotations, fmt="", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, cbar_kws={"label": "Conditioning gain (pp)"}, ax=axis)
    mark_significant_heatmap_cells(axis, conditioning_significance)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, conditioning_table)
    axis.axvline(3, color="white", linewidth=2.5)
    axis.set(title="Active card-conditioning gains by contrast", xlabel="Metric and contrast", ylabel="Model")
    axis.tick_params(axis="x", rotation=0, labelsize=9)
    add_significance_caption(figure)
    figure.tight_layout(rect=[0, .05, 1, 1])
    save_panel_figure(figure, "panel_12", "plot_quotation_and_search_scope_contrasts.png")
    plt.show()
contact_cost_gain = active_search[active_search["metric"] == "mean_cost_to_first_target_contact_seconds"][["model", "card_conditioning_gain", "card_conditioning_gain_ci_95_lower", "card_conditioning_gain_ci_95_upper", "supported_reels_conditioning"]].rename(columns={"card_conditioning_gain": "contact_cost_gain_seconds", "card_conditioning_gain_ci_95_lower": "contact_cost_gain_ci_95_lower_seconds", "card_conditioning_gain_ci_95_upper": "contact_cost_gain_ci_95_upper_seconds"})
save_panel_table(contact_cost_gain, "panel_12", "table_contact_cost_conditioning_effect.csv")
display(Markdown("**Active card-conditioning effect on contact cost (seconds; negative is better, N/A means no contacted-target denominator)**")); display(contact_cost_gain)


In [ ]:
display(Markdown("**Alternative line view — quote-form by search-scope interaction**"))
from matplotlib.lines import Line2D
quote_line_metrics = {"answer_score": "Answer", "localization_pass": "Localization pass", "temporal_f1": "Temporal F1", "target_contact_rate": "Target contact"}
quote_line = cell_means.groupby(["model", "quote_form", "search_scope"], dropna=False)[list(quote_line_metrics)].mean().reset_index()
scope_styles = {"global": "-", "card-conditioned": "--"}
figure, axes = plt.subplots(1, len(quote_line_metrics), figsize=(16, 4.8), sharex=True, sharey=True)
for axis, (metric, metric_label) in zip(axes, quote_line_metrics.items()):
    for model in alternative_model_order:
        for scope, linestyle in scope_styles.items():
            block = quote_line[(quote_line["model"] == model) & (quote_line["search_scope"] == scope)].set_index("quote_form").reindex(["exact", "partial"])
            if block[metric].isna().any(): continue
            axis.plot([0, 1], 100 * block[metric].to_numpy(), marker="o", markersize=4, linewidth=1.35, linestyle=linestyle, color=alternative_model_colors[model], alpha=.9)
    axis.set(title=metric_label, xticks=[0, 1], xticklabels=["Exact quote", "Partial quote"], ylim=(-3, 103)); axis.grid(axis="y", alpha=.2)
axes[0].set_ylabel("Score / rate (%)")
model_handles = [Line2D([0], [0], color=alternative_model_colors[model], marker="o", linewidth=1.4, label=short_model_label(model)) for model in alternative_model_order if model in set(quote_line["model"])]
scope_handles = [Line2D([0], [0], color="#444444", linestyle=style, linewidth=1.5, label=scope.title()) for scope, style in scope_styles.items()]
figure.legend(handles=model_handles + scope_handles, loc="lower center", ncol=min(5, len(model_handles) + len(scope_handles)), frameon=False)
figure.suptitle("Panel 12 alternative: exact/partial quote interaction with search scope", y=.99)
figure.text(.5, .92, "Formal Holm-corrected averaged contrasts remain in the original heatmap above.", ha="center", fontsize=8.5)
figure.tight_layout(rect=[0, .15, 1, .86])
save_panel_figure(figure, "panel_12", "plot_quotation_search_interaction_alternative.png")
plt.show()


### D. Combined Diagnostics

### Panel 13. Evidence Acquisition, Efficiency, and Belief Dissociation

Block A uses evidence reach—not the headline efficiency-aware acquisition score—to separate failure to find evidence from failure to represent it. The declared diagnosis is accompanied by a stacked lenient/declared/strict threshold-sensitivity summary, so category changes near the cut-points remain visible. These categories are descriptive conventions, not validated natural classes. In both scatter views, Tracks 1A and 1B are first averaged within model and displayed as one acoustic-mapping facet; each plotted acoustic metric requires support from both tracks. Track 1C and Track 2 remain separate. All scatter facets use common `x = 0.5` and `y = 0.5` dashed references. The efficiency frontier jointly plots budget used and replay rate, with evidence reach encoded by point size. Track 2A.3 is an absent-utterance task, so it has no present-target evidence-coverage denominator and is displayed as gray `N/A` in that specific heatmap. Labeled red stars mark the available ideal reference points: `(1, 1)` for reach/belief and `(0, 0)` for the two cost axes.


In [ ]:
t1_combined = main_slice(episode_metrics, conditions=["active"])[["model", "track", "master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"]].copy()
t2_combined = track2_family_macro(track2_active, ["master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"], group_cols=("model",), include_overall=True)
t2_combined = t2_combined[t2_combined["family"] == "Track 2 overall"].copy(); t2_combined["track"] = "2"
combined = pd.concat([t1_combined.groupby(["model", "track"], dropna=False)[["master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"]].mean().reset_index(), t2_combined[["model", "track", "master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"]]], ignore_index=True)
combined["diagnosis"] = np.select(
    [(combined["evidence_reach_score"] < EVIDENCE_THRESHOLD) & (combined["temporal_belief_score"] < LOW_SCORE_THRESHOLD),
     (combined["evidence_reach_score"] >= EVIDENCE_THRESHOLD) & (combined["temporal_belief_score"] < LOW_SCORE_THRESHOLD),
     (combined["evidence_reach_score"] >= EVIDENCE_THRESHOLD) & (combined["temporal_belief_score"] >= LOW_SCORE_THRESHOLD)],
    ["acquisition-limited", "belief-limited", "robust"], default="inspect priors/metric mismatch")
save_panel_table(combined, "panel_13", "table_combined_evidence_efficiency_belief_diagnostics.csv")
display(percent_table(combined, ["master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"]))

combined_threshold_frames = []
for threshold_scenario, thresholds in DIAGNOSTIC_THRESHOLD_SCENARIOS.items():
    scenario_combined = combined.drop(columns=["diagnosis"]).copy()
    scenario_combined["threshold_scenario"] = threshold_scenario
    scenario_combined["evidence_threshold"] = thresholds["evidence"]
    scenario_combined["belief_threshold"] = thresholds["low_score"]
    scenario_combined["diagnosis"] = np.select(
        [
            (scenario_combined["evidence_reach_score"] < thresholds["evidence"]) & (scenario_combined["temporal_belief_score"] < thresholds["low_score"]),
            (scenario_combined["evidence_reach_score"] >= thresholds["evidence"]) & (scenario_combined["temporal_belief_score"] < thresholds["low_score"]),
            (scenario_combined["evidence_reach_score"] >= thresholds["evidence"]) & (scenario_combined["temporal_belief_score"] >= thresholds["low_score"]),
        ],
        ["acquisition-limited", "belief-limited", "robust"],
        default="inspect priors/metric mismatch",
    )
    combined_threshold_frames.append(scenario_combined)
combined_threshold_sensitivity = pd.concat(combined_threshold_frames, ignore_index=True)
save_panel_table(combined_threshold_sensitivity, "panel_13", "table_evidence_belief_threshold_sensitivity.csv")
combined_threshold_summary = combined_threshold_sensitivity.groupby(["threshold_scenario", "diagnosis"], observed=False).size().rename("n_model_tracks").reset_index()
combined_threshold_summary["share"] = combined_threshold_summary["n_model_tracks"] / combined_threshold_summary.groupby("threshold_scenario")["n_model_tracks"].transform("sum")
display(percent_table(combined_threshold_summary, ["share"]))
combined_diagnosis_order = [diagnosis for diagnosis in ["acquisition-limited", "belief-limited", "robust", "inspect priors/metric mismatch"] if diagnosis in set(combined_threshold_summary["diagnosis"])]
combined_sensitivity_plot = combined_threshold_summary.pivot(index="threshold_scenario", columns="diagnosis", values="share").reindex(index=DIAGNOSTIC_THRESHOLD_SCENARIOS, columns=combined_diagnosis_order, fill_value=0).fillna(0)
axis = combined_sensitivity_plot.plot(kind="bar", stacked=True, figsize=(7.6, 4.4), color=sns.color_palette("Set2", len(combined_diagnosis_order)))
axis.set(title="Evidence-belief diagnosis sensitivity to threshold convention", xlabel="Threshold convention", ylabel="Share of model-track cells", ylim=(0, 1))
axis.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); axis.tick_params(axis="x", rotation=0)
axis.legend(title="Diagnosis", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
axis.figure.tight_layout()
save_panel_figure(axis.figure, "panel_13", "plot_evidence_belief_threshold_sensitivity.png")
plt.show()
t2_efficiency_metrics = ["budget_used", "replay_rate", "unique_reel_coverage", "mean_target_evidence_coverage", "target_contact_rate"]
t2_efficiency_detail = track2_family_macro(track2_active, t2_efficiency_metrics, group_cols=("model",))
t2_efficiency_long = t2_efficiency_detail.melt(id_vars=["model", "family", "n_tasks", "n_reels"], value_vars=t2_efficiency_metrics, var_name="metric", value_name="value")
metric_support_rows = []
for (model, family), group in track2_active.groupby(["model", "family"], dropna=False):
    for metric in t2_efficiency_metrics:
        supported = group[pd.to_numeric(group[metric], errors="coerce").notna()]
        metric_support_rows.append({"model": model, "family": family, "metric": metric, "supported_tasks": supported["task_id"].nunique(), "supported_reels": supported["reel_id"].nunique()})
t2_efficiency_long = t2_efficiency_long.merge(pd.DataFrame(metric_support_rows), on=["model", "family", "metric"], validate="one_to_one")
save_panel_table(t2_efficiency_long, "panel_13", "table_track2_efficiency_components.csv")
display(Markdown("**Track 2 efficiency components with metric-specific support**"))
display(percent_table(t2_efficiency_long, ["value"]))
coverage_specs = [("unique_reel_coverage", "Unique reel coverage"), ("mean_target_evidence_coverage", "Present-target evidence coverage\n(2A.3 N/A)")]
coverage_tables = []
coverage_model_order = sorted(t2_efficiency_long["model"].unique(), key=short_model_label)
for metric, title in coverage_specs:
    table = t2_efficiency_long[t2_efficiency_long["metric"] == metric].pivot(index="model", columns="family", values="value").reindex(index=coverage_model_order, columns=TRACK2_FAMILIES)
    table.index = table.index.map(short_model_label)
    coverage_tables.append((title, table))
figure, axes = plt.subplots(1, 2, figsize=(13.5, 4.4), gridspec_kw={"width_ratios": [5, 5]})
colorbar_axis = figure.add_axes([0.92, 0.18, 0.018, 0.64])
for index, (title, table) in enumerate(coverage_tables):
    sns.heatmap(100 * table, mask=table.isna(), annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100, ax=axes[index], cbar=index == 0, cbar_ax=colorbar_axis if index == 0 else None)
    axes[index].set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axes[index], table)
    axes[index].set(title=title, xlabel="Family", ylabel="Model" if index == 0 else "")
    axes[index].tick_params(axis="x", rotation=20)
    if index == 1: axes[index].tick_params(axis="y", labelleft=False)
colorbar_axis.set_ylabel("Coverage (%)")
figure.suptitle("Track 2 acquisition coverage", y=0.98)
figure.subplots_adjust(left=0.09, right=0.90, bottom=0.18, top=0.82, wspace=0.10)
save_panel_figure(figure, "panel_13", "plot_track2_acquisition_coverage.png")
plt.show()
panel13_plot_metrics = ["master_score", "evidence_reach_score", "temporal_belief_score", "budget_used", "replay_rate"]
panel13_acoustic_source = combined[combined["track"].isin(["1a", "1b"])].copy()
assert panel13_acoustic_source.groupby("model")["track"].nunique().eq(2).all(), "Panel 13 acoustic scatter requires both Track 1A and Track 1B"
panel13_acoustic_means = panel13_acoustic_source.groupby("model", dropna=False)[panel13_plot_metrics].mean()
panel13_acoustic_support = panel13_acoustic_source.groupby("model", dropna=False)[panel13_plot_metrics].count()
panel13_acoustic = panel13_acoustic_means.where(panel13_acoustic_support.eq(2)).reset_index()
panel13_acoustic["track"] = "1a/1b"
panel13_acoustic["diagnosis"] = np.select(
    [(panel13_acoustic["evidence_reach_score"] < EVIDENCE_THRESHOLD) & (panel13_acoustic["temporal_belief_score"] < LOW_SCORE_THRESHOLD),
     (panel13_acoustic["evidence_reach_score"] >= EVIDENCE_THRESHOLD) & (panel13_acoustic["temporal_belief_score"] < LOW_SCORE_THRESHOLD),
     (panel13_acoustic["evidence_reach_score"] >= EVIDENCE_THRESHOLD) & (panel13_acoustic["temporal_belief_score"] >= LOW_SCORE_THRESHOLD)],
    ["acquisition-limited", "belief-limited", "robust"], default="inspect priors/metric mismatch")
combined_scatter = pd.concat([panel13_acoustic, combined[combined["track"].isin(["1c", "2"])][["model", "track", *panel13_plot_metrics, "diagnosis"]]], ignore_index=True)
save_panel_table(combined_scatter, "panel_13", "table_scatter_plot_aggregates.csv")
panel13_track_order = ["1a/1b", "1c", "2"]
panel13_track_titles = {"1a/1b": "Tracks 1A/1B mean", "1c": "Track 1C", "2": "Track 2"}
grid = sns.relplot(data=combined_scatter.dropna(subset=["evidence_reach_score", "temporal_belief_score", "master_score", "diagnosis"]), x="evidence_reach_score", y="temporal_belief_score", size="master_score", hue="diagnosis", col="track", col_order=panel13_track_order, col_wrap=3, height=3.2, aspect=.95, sizes=(45, 240), alpha=.85)
grid.set(xlim=(0, 1), ylim=(0, 1), xlabel="Evidence reach", ylabel="Temporal belief"); grid.set_titles("")
for ax, track in zip(grid.axes.flat, panel13_track_order):
    ax.set_title(panel13_track_titles[track])
    ax.axvline(0.5, color="gray", ls="--"); ax.axhline(0.5, color="gray", ls="--"); add_ideal_star(ax, 1.0, 1.0)
    for collection in ax.collections:
        collection.set_clip_on(False)
        collection.set_zorder(max(collection.get_zorder(), 10))
add_ideal_star_legend(grid, "Ideal (1, 1)", next_section_label="master_score")
grid.figure.subplots_adjust(wspace=0.16)
save_panel_figure(grid.figure, "panel_13", "plot_evidence_belief_dissociation.png")
plt.show()
pareto_source = combined_scatter.dropna(subset=["budget_used", "evidence_reach_score"]).copy()
pareto_source["track"] = pd.Categorical(pareto_source["track"], panel13_track_order, ordered=True)
pareto_source["Model"] = pareto_source["model"].map(short_model_label)
pareto_source["Evidence reach"] = pareto_source["evidence_reach_score"].clip(0, 1)
pareto_data = pareto_source.dropna(subset=["replay_rate"]).copy()
missing_replay = pareto_source[pareto_source["replay_rate"].isna()].copy()
pareto_model_order = sorted(pareto_source["Model"].unique())
pareto_palette = model_palette(pareto_model_order)
grid = sns.relplot(data=pareto_data, x="budget_used", y="replay_rate", hue="Model", hue_order=pareto_model_order, palette=pareto_palette, size="Evidence reach", col="track", col_order=panel13_track_order, col_wrap=3, height=3.2, aspect=.95, sizes=(40, 240), alpha=.85)
grid.set(xlim=(0, 1), ylim=(0, 1), xlabel="", ylabel="Replay rate"); grid.set_titles("")
for ax, track in zip(grid.axes.flat, panel13_track_order):
    ax.set_title(panel13_track_titles[track])
    ax.axvline(0.5, color="gray", ls="--"); ax.axhline(0.5, color="gray", ls="--")
    add_ideal_star(ax, 0.0, 0.0)
    for collection in ax.collections:
        collection.set_clip_on(False)
        collection.set_zorder(max(collection.get_zorder(), 10))
    missing_track = missing_replay[missing_replay["track"] == track]
    if not missing_track.empty:
        marker_sizes = 40 + 200 * missing_track["Evidence reach"].to_numpy(dtype=float)
        marker_colors = [pareto_palette[model] for model in missing_track["Model"]]
        ax.scatter(missing_track["budget_used"], np.full(len(missing_track), -0.16), transform=ax.get_xaxis_transform(), marker="X", s=marker_sizes, c=marker_colors, edgecolors="white", linewidth=.6, alpha=.65, clip_on=False, zorder=15)
        ax.text(-0.02, -0.16, "Replay N/A", transform=ax.transAxes, ha="right", va="center", fontsize=7)
add_ideal_star_legend(grid, "Ideal costs (0, 0)", next_section_label="Evidence reach")
if grid._legend is not None: grid._legend.set_frame_on(False)
grid.figure.suptitle("Efficiency frontier: lower costs and greater evidence reach", y=1.03)
grid.figure.supxlabel("Budget used", y=0.03)
grid.figure.subplots_adjust(wspace=0.16, bottom=0.24)
save_panel_figure(grid.figure, "panel_13", "plot_efficiency_frontier.png")
plt.show()


### Panel 14. Uncertainty Awareness and Supported Commitment

Track 1 retains only episodes where both known and unknown scores are defined. These are different task constructs rather than symmetric matched items, so known-minus-unknown is a descriptive contrast and cannot isolate overclaiming by itself. Track 2 separates positive-claim support, factual absence accuracy, evidence-supported absence, true-but-undercovered absence claims, final calibration, and unresolved-ledger rate. An absence claim is evidence-supported only when the target is truly absent and claim-time unique reel coverage is at least 80%; false and true-but-undercovered claims remain distinct. The aligned heatmaps suppress repeated nominal support counts but retain reduced support and `N/A (n=0)`. Claim-support cells are `N/A` when a model made no claim of that type; calibration is `N/A` when correctness/confidence was not jointly evaluable. Up-arrows mark higher-is-better scores and down-arrows mark lower-is-better failure rates.


In [ ]:
track1_uncertainty_source = main_slice(episode_metrics, conditions=["active"])
track1_uncertainty_base = track1_uncertainty_source.groupby(["model", "track"], dropna=False).size().rename("total_episodes").reset_index()
track1_uncertainty_source["paired_cluster_id"] = track1_paired_cluster_id(track1_uncertainty_source)
track1_uncertainty_paired = track1_uncertainty_source.dropna(subset=["known_score", "unknown_score", "paired_cluster_id"])
track1_uncertainty_rows = []
for (model, track), group in track1_uncertainty_paired.groupby(["model", "track"], dropna=False):
    difference = group["known_score"] - group["unknown_score"]
    estimate, lower, upper, cluster_count, _ = clustered_mean_ci(difference, group["paired_cluster_id"])
    p_value = cluster_sign_flip_p_value(difference, group["paired_cluster_id"])
    track1_uncertainty_rows.append({
        "model": model, "track": track,
        "known_score": group["known_score"].mean(),
        "unknown_score": group["unknown_score"].mean(),
        "known_minus_unknown_bias": estimate,
        "known_minus_unknown_bias_ci_95_lower": lower,
        "known_minus_unknown_bias_ci_95_upper": upper,
        "known_minus_unknown_bias_p_value": p_value,
        "supported_episodes": group["episode_id"].nunique(),
        "cluster_count": cluster_count,
    })
track1_uncertainty = pd.DataFrame(track1_uncertainty_rows)
track1_uncertainty = track1_uncertainty_base.merge(track1_uncertainty, on=["model", "track"], how="left"); track1_uncertainty["supported_episodes"] = track1_uncertainty["supported_episodes"].fillna(0).astype(int)
track1_uncertainty["cluster_count"] = track1_uncertainty["cluster_count"].fillna(0).astype(int)
track1_uncertainty["known_minus_unknown_bias_holm_p_value"] = holm_adjusted_p_values(track1_uncertainty["known_minus_unknown_bias_p_value"])
track1_uncertainty["known_minus_unknown_bias_significant"] = track1_uncertainty["known_minus_unknown_bias_holm_p_value"].lt(SIGNIFICANCE_ALPHA)
save_panel_table(track1_uncertainty, "panel_14", "table_track1_known_unknown_scores.csv")
display(Markdown("**Track 1 known/unknown block**")); display(percent_table(track1_uncertainty, ["known_score", "unknown_score", "known_minus_unknown_bias", "known_minus_unknown_bias_ci_95_lower", "known_minus_unknown_bias_ci_95_upper"]))
track1_uncertainty_table = track1_uncertainty.pivot(index="model", columns="track", values="known_minus_unknown_bias")
track1_uncertainty_support = track1_uncertainty.pivot(index="model", columns="track", values="supported_episodes")
track1_uncertainty_significance = track1_uncertainty.pivot(index="model", columns="track", values="known_minus_unknown_bias_significant").reindex(index=track1_uncertainty_table.index, columns=track1_uncertainty_table.columns)
track1_uncertainty_table.index = track1_uncertainty_table.index.map(short_model_label); track1_uncertainty_support.index = track1_uncertainty_support.index.map(short_model_label)
track1_uncertainty_significance.index = track1_uncertainty_table.index
vmax = max(.05, np.nanmax(np.abs(track1_uncertainty_table.to_numpy())))

t2_uncertainty = track2_family_macro(track2_active, ["found_claim_support_score", "absence_factual_accuracy_score", "absence_claim_support_score", "undercovered_true_absence_claim_rate", "calibration_score", "final_unresolved_ledger_rate"], group_cols=("model",), include_overall=True)
t2_uncertainty = t2_uncertainty[t2_uncertainty["family"] == "Track 2 overall"]
t2_uncertainty_metrics = ["found_claim_support_score", "absence_factual_accuracy_score", "absence_claim_support_score", "undercovered_true_absence_claim_rate", "calibration_score", "final_unresolved_ledger_rate"]
t2_uncertainty_labels = {"found_claim_support_score": "Found support ↑", "absence_factual_accuracy_score": "Absence factual ↑", "absence_claim_support_score": "Absence supported ↑", "undercovered_true_absence_claim_rate": "True absence undercovered ↓", "calibration_score": "Calibration ↑", "final_unresolved_ledger_rate": "Unresolved ledger ↓"}
t2_uncertainty_support = track2_active.groupby("model")[t2_uncertainty_metrics].count()
save_panel_table(t2_uncertainty, "panel_14", "table_track2_claim_support_and_calibration.csv")
display(Markdown("**Track 2 claim-support and calibration block**")); display(percent_table(t2_uncertainty, t2_uncertainty_metrics))
t2_uncertainty_table = t2_uncertainty.set_index("model")[t2_uncertainty_metrics].rename(columns=t2_uncertainty_labels)
t2_uncertainty_support = t2_uncertainty_support.rename(columns=t2_uncertainty_labels)
t2_uncertainty_table.index = t2_uncertainty_table.index.map(short_model_label); t2_uncertainty_support.index = t2_uncertainty_support.index.map(short_model_label)
uncertainty_model_order = sorted(set(track1_uncertainty_table.index) | set(t2_uncertainty_table.index))
track1_uncertainty_table = track1_uncertainty_table.reindex(uncertainty_model_order); track1_uncertainty_support = track1_uncertainty_support.reindex(uncertainty_model_order).fillna(0); track1_uncertainty_significance = track1_uncertainty_significance.reindex(uncertainty_model_order).fillna(False)
t2_uncertainty_table = t2_uncertainty_table.reindex(uncertainty_model_order); t2_uncertainty_support = t2_uncertainty_support.reindex(uncertainty_model_order).fillna(0)
t2_uncertainty_table = t2_uncertainty_table.rename(columns={"Found support ↑": "Found\nsupport ↑", "Absence factual ↑": "Absence\nfactual ↑", "Absence supported ↑": "Absence\nsupported ↑", "True absence undercovered ↓": "True absence\nundercovered ↓", "Unresolved ledger ↓": "Unresolved\nledger ↓"})
t2_uncertainty_support = t2_uncertainty_support.rename(columns={"Found support ↑": "Found\nsupport ↑", "Absence factual ↑": "Absence\nfactual ↑", "Absence supported ↑": "Absence\nsupported ↑", "True absence undercovered ↓": "True absence\nundercovered ↓", "Unresolved ledger ↓": "Unresolved\nledger ↓"})
figure, axes = plt.subplots(1, 2, figsize=(15.5, 5.3), gridspec_kw={"width_ratios": [3, 6]})
track1_annotations = selective_support_annotations(track1_uncertainty_table, track1_uncertainty_support)
sns.heatmap(100 * track1_uncertainty_table, mask=track1_uncertainty_table.isna(), annot=track1_annotations, fmt="", cmap="vlag", center=0, vmin=-100 * vmax, vmax=100 * vmax, cbar_kws={"label": "Known - unknown (pp)", "shrink": .85}, ax=axes[0])
mark_significant_heatmap_cells(axes[0], track1_uncertainty_significance)
axes[0].set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axes[0], track1_uncertainty_table, track1_uncertainty_support)
axes[0].set(title="A. Track 1 known-minus-unknown bias", xlabel="Track", ylabel="Model"); axes[0].tick_params(axis="x", rotation=0)
track2_annotations = selective_support_annotations(t2_uncertainty_table, t2_uncertainty_support)
sns.heatmap(100 * t2_uncertainty_table, mask=t2_uncertainty_table.isna(), annot=track2_annotations, fmt="", cmap="YlGnBu", vmin=0, vmax=100, cbar_kws={"label": "Score / rate (%)", "shrink": .85}, ax=axes[1])
axes[1].set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axes[1], t2_uncertainty_table, t2_uncertainty_support)
axes[1].set(title="B. Track 2 supported commitment and uncertainty", xlabel="Metric", ylabel=""); axes[1].tick_params(axis="x", rotation=0, labelsize=9); axes[1].tick_params(axis="y", labelleft=False)
figure.suptitle("Uncertainty awareness and supported commitment", y=0.99)
add_significance_caption(figure, "all displayed Block A contrast cells")
figure.tight_layout(rect=[0, .05, 1, .97])
save_panel_figure(figure, "panel_14", "plot_uncertainty_awareness_and_supported_commitment.png")
plt.show()


In [ ]:
display(Markdown("**Alternative line view — Track 1 known and unknown scores**"))
figure, axes = plt.subplots(1, 3, figsize=(11.5, 4.8), sharex=True, sharey=True)
for axis, track in zip(axes, ["1a", "1b", "1c"]):
    block = track1_uncertainty[track1_uncertainty["track"] == track]
    for row in block.itertuples(index=False):
        if pd.isna(row.known_score) or pd.isna(row.unknown_score): continue
        axis.plot([0, 1], 100 * np.array([row.known_score, row.unknown_score]), marker="o", markersize=4.5, linewidth=1.4, color=alternative_model_colors[row.model], label=short_model_label(row.model))
        if row.known_minus_unknown_bias_significant:
            axis.scatter(1, 100 * row.unknown_score, s=32, facecolor="black", edgecolor="white", linewidth=.7, zorder=20)
    axis.set(title=f"Track {track.upper()}", xticks=[0, 1], xticklabels=["Known", "Unknown"], ylim=(-3, 103)); axis.grid(axis="y", alpha=.2)
axes[0].set_ylabel("Score (%)")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
figure.suptitle("Panel 14 alternative: known-to-unknown score slopes", y=.99)
figure.text(.5, .92, "● Holm-adjusted p < 0.05 for the corresponding known-minus-unknown contrast", ha="center", fontsize=8.5)
figure.tight_layout(rect=[0, .14, 1, .86])
save_panel_figure(figure, "panel_14", "plot_known_unknown_score_slopes_alternative.png")
plt.show()


### Panel 15. Diagnostic Failure Flags and Earliest Flagged Stage

Two complementary descriptive views are reported. The primary stacked bars assign each unit to the earliest applicable flag in a declared diagnostic precedence order from evaluability through stability/revision; this ordering is not a causal sequence, and an earlier flag merely masks later flags in that view. The companion grid preserves all co-occurring failure rates, with exact applicable-unit denominators and total units in the table. `N/A` means the flag is structurally inapplicable or no eligible event occurred—for example, no FIND claim, no revision opportunity, or no valid component score. Track 1A/1B stability is omitted because probe changes conflate newly acquired evidence with stability. A lenient/declared/strict sensitivity analysis shows how general flag rates change under nearby threshold conventions.


In [ ]:
def nullable_compare(series, comparison):
    numeric = pd.to_numeric(series, errors="coerce")
    return comparison(numeric).where(numeric.notna(), pd.NA).astype("boolean")


def nullable_any(*flags):
    frame = pd.concat([flag.astype("boolean") for flag in flags], axis=1)
    result = pd.Series(pd.NA, index=frame.index, dtype="boolean")
    result.loc[frame.eq(True).any(axis=1)] = True
    result.loc[frame.notna().all(axis=1) & frame.eq(False).all(axis=1)] = False
    return result


def nullable_all(*flags):
    frame = pd.concat([flag.astype("boolean") for flag in flags], axis=1)
    result = pd.Series(pd.NA, index=frame.index, dtype="boolean")
    result.loc[frame.eq(False).any(axis=1)] = False
    result.loc[frame.notna().all(axis=1) & frame.eq(True).all(axis=1)] = True
    return result


def nullable_any_applicable(*flags):
    frame = pd.concat([flag.astype("boolean") for flag in flags], axis=1)
    result = pd.Series(pd.NA, index=frame.index, dtype="boolean")
    result.loc[frame.eq(True).any(axis=1)] = True
    result.loc[frame.notna().any(axis=1) & ~frame.eq(True).any(axis=1)] = False
    return result


def below(series, threshold):
    return nullable_compare(series, lambda values: values < threshold)


def at_least(series, threshold):
    return nullable_compare(series, lambda values: values >= threshold)


def equal_to(series, value):
    return nullable_compare(series, lambda values: values == value)


# General diagnostic flags preserve missing component support and expose threshold sensitivity.
t1_failure_base = main_slice(episode_metrics, conditions=["active"]).copy()
trajectory_for_failure = drift[["model", "track", "episode_id", "drift"]].copy()
assert not trajectory_for_failure.duplicated(["model", "track", "episode_id"], keep=False).any(), "Duplicate Track 1 trajectory episodes"
t1_failure_base = t1_failure_base.merge(trajectory_for_failure, on=["model", "track", "episode_id"], how="left", validate="one_to_one")
passive_answer = main_slice(episode_metrics, conditions=["full_context"]).groupby(["model", "track"], dropna=False)["answer_score"].mean().rename("passive_answer_score").reset_index()
t1_failure_base = t1_failure_base.merge(passive_answer, on=["model", "track"], how="left")
t1_failure_base["domain"] = np.where(t1_failure_base["track"].isin(["1a", "1b"]), "Controlled Acoustic Temporal Mapping", "Controlled Conversational Belief Updating")
t2_failure_base = track2_metrics[track2_metrics["condition"] == "active"].copy()
t2_failure_base["domain"] = "Long-Horizon Conversational Navigation"

def apply_t1_general_flags(frame, thresholds):
    out = frame.copy()
    out["protocol_or_evaluability"] = nullable_any((~out["run_evaluable"]).astype("boolean"), below(out["protocol_component_score"], thresholds["protocol"]))
    out["evidence_acquisition"] = below(out["evidence_reach_score"], thresholds["evidence"])
    out["acquisition_efficiency"] = nullable_any_applicable(at_least(out["replay_rate"], thresholds["cost"]), at_least(out["redundant_query_rate"], thresholds["cost"]), at_least(out["silent_waste"], thresholds["cost"]))
    out["belief_construction"] = below(out["temporal_belief_score"], thresholds["low_score"])
    out["uncertainty_or_claim_support"] = below(out["uncertainty_score"], thresholds["low_score"])
    out["final_answer_integration"] = nullable_all(at_least(out["temporal_belief_score"], thresholds["low_score"]), below(out["answer_score"], thresholds["low_score"]))
    drift_failure = at_least(out["drift"], thresholds["drift"]).where(out["track"].eq("1c"), pd.NA)
    inertia_failure = at_least(out["stale_answer_rate_given_evidence"], thresholds["stale"]).where(out["track"].eq("1c"), pd.NA)
    out["stability_or_revision"] = nullable_any_applicable(drift_failure, inertia_failure)
    return out

def apply_t2_general_flags(frame, thresholds):
    out = frame.copy()
    out["protocol_or_evaluability"] = nullable_any((~out["run_evaluable"]).astype("boolean"), below(out["protocol_component_score"], thresholds["protocol"]))
    out["evidence_acquisition"] = below(out["evidence_reach_score"], thresholds["evidence"])
    out["acquisition_efficiency"] = nullable_any(at_least(out["replay_rate"], thresholds["cost"]), nullable_all(at_least(out["budget_used"], thresholds["high_budget"]), below(out["evidence_reach_score"], thresholds["budget_reach"])))
    out["belief_construction"] = below(out["temporal_belief_score"], thresholds["low_score"])
    out["uncertainty_or_claim_support"] = below(out["uncertainty_score"], thresholds["low_score"])
    out["final_answer_integration"] = nullable_all(at_least(out["temporal_belief_score"], thresholds["low_score"]), below(out["answer_score"], thresholds["low_score"]))
    out["stability_or_revision"] = below(out.get("successful_revision_rate"), thresholds["low_score"])
    return out

t1_failure = apply_t1_general_flags(t1_failure_base, DECLARED_THRESHOLDS)
t2_failure = apply_t2_general_flags(t2_failure_base, DECLARED_THRESHOLDS)

# Track 2 family-specific flags retain non-complete rows but use nullable applicability.

t2_failure["unsupported_found_claim"] = nullable_compare(t2_failure.get("unsupported_found_claim_rate"), lambda values: values > 0).where(t2_failure["found_claim_count"].gt(0), pd.NA)
t2_failure["false_absence_claim"] = nullable_compare(t2_failure.get("false_absence_claim_rate"), lambda values: values > 0).where(t2_failure["absence_claim_count"].gt(0), pd.NA)
t2_failure["undercovered_true_absence_claim"] = nullable_compare(t2_failure.get("undercovered_true_absence_claim_rate"), lambda values: values > 0).where(t2_failure["absence_claim_count"].gt(0), pd.NA)
t2_failure["scene_or_boundary_failure"] = nullable_any_applicable(
    below(t2_failure.get("localization_pass"), 1.0), below(t2_failure.get("scene_identity_accuracy"), 1.0), below(t2_failure.get("utterance_localization_accuracy"), 1.0)
)
t2_failure["ordering_failure"] = below(t2_failure.get("exact_order_accuracy"), 1.0)
t2_failure["scene_assignment_failure"] = nullable_any_applicable(below(t2_failure.get("scene_card_assignment_accuracy"), 1.0), below(t2_failure.get("card_to_scene_assignment_accuracy"), 1.0))
ledger_exact = nullable_all(
    equal_to(t2_failure.get("final_ledger_target_accuracy"), 1.0), equal_to(t2_failure.get("final_ledger_status_accuracy"), 1.0), equal_to(t2_failure.get("final_ledger_complete_accuracy"), 1.0)
)
t2_failure["correct_ledger_wrong_answer"] = nullable_all(ledger_exact, below(t2_failure["primary_correct"], 1.0))
t2_failure["incomplete_2b2"] = below(t2_failure.get("all_components_complete_accuracy"), 1.0).where(t2_failure["family"].eq("2B.2"), pd.NA)

stage_columns = ["protocol_or_evaluability", "evidence_acquisition", "acquisition_efficiency", "belief_construction", "uncertainty_or_claim_support", "final_answer_integration", "stability_or_revision"]
def earliest_flagged_stage(row):
    return next((stage for stage in stage_columns if pd.notna(row.get(stage)) and bool(row.get(stage))), "no applicable general flag")
t1_failure["earliest_flagged_stage"] = t1_failure.apply(earliest_flagged_stage, axis=1)
t2_failure["earliest_flagged_stage"] = t2_failure.apply(earliest_flagged_stage, axis=1)
first_rates_t1 = t1_failure.groupby(["model", "domain", "earliest_flagged_stage"], dropna=False).size().rename("units").reset_index()
first_rates_t2 = t2_failure.groupby(["model", "domain", "earliest_flagged_stage"], dropna=False).size().rename("units").reset_index()
first_rates = pd.concat([first_rates_t1, first_rates_t2], ignore_index=True)
first_rates["rate"] = first_rates["units"] / first_rates.groupby(["model", "domain"])["units"].transform("sum")
save_panel_table(first_rates, "panel_15", "table_earliest_flagged_stage.csv")
display(Markdown("**View 1 — Earliest flagged stage under the declared diagnostic precedence**")); display(percent_table(first_rates, ["rate"]))

t2_specific = ["unsupported_found_claim", "false_absence_claim", "undercovered_true_absence_claim", "scene_or_boundary_failure", "ordering_failure", "scene_assignment_failure", "correct_ledger_wrong_answer", "incomplete_2b2"]
def summarize_nullable_flags(frame, flag_columns):
    records = []
    for (model, domain), group in frame.groupby(["model", "domain"], dropna=False):
        for flag in flag_columns:
            applicable = group[flag].dropna()
            records.append({"model": model, "domain": domain, "failure_flag": flag, "failure_rate": applicable.astype(float).mean() if len(applicable) else np.nan, "applicable_units": len(applicable), "total_units": len(group)})
    return pd.DataFrame(records)
all_flags = pd.concat([summarize_nullable_flags(t1_failure, stage_columns), summarize_nullable_flags(t2_failure, stage_columns + t2_specific)], ignore_index=True)
save_panel_table(all_flags, "panel_15", "table_all_applicable_failure_flags.csv")
display(Markdown("**View 2 — All applicable failure flags with denominators**")); display(percent_table(all_flags, ["failure_rate"]))

threshold_sensitivity_blocks = []
for scenario, thresholds in DIAGNOSTIC_THRESHOLD_SCENARIOS.items():
    scenario_t1 = apply_t1_general_flags(t1_failure_base, thresholds)
    scenario_t2 = apply_t2_general_flags(t2_failure_base, thresholds)
    scenario_flags = pd.concat([summarize_nullable_flags(scenario_t1, stage_columns), summarize_nullable_flags(scenario_t2, stage_columns)], ignore_index=True)
    scenario_flags.insert(0, "threshold_scenario", scenario)
    for threshold_name, threshold_value in thresholds.items():
        scenario_flags[f"threshold_{threshold_name}"] = threshold_value
    threshold_sensitivity_blocks.append(scenario_flags)
threshold_sensitivity = pd.concat(threshold_sensitivity_blocks, ignore_index=True)
save_panel_table(threshold_sensitivity, "panel_15", "table_general_failure_threshold_sensitivity.csv")
display(Markdown("**Threshold sensitivity — general diagnostic flags**")); display(percent_table(threshold_sensitivity, ["failure_rate"]))
failure_flag_labels = {
    "protocol_or_evaluability": "Protocol / evaluability", "evidence_acquisition": "Evidence acquisition",
    "acquisition_efficiency": "Acquisition efficiency", "belief_construction": "Belief construction",
    "uncertainty_or_claim_support": "Uncertainty / support", "final_answer_integration": "Answer integration",
    "stability_or_revision": "Stability / revision", "unsupported_found_claim": "Unsupported FIND",
    "false_absence_claim": "Factually false absence", "undercovered_true_absence_claim": "True absence undercovered", "scene_or_boundary_failure": "Scene / boundary",
    "ordering_failure": "Ordering", "scene_assignment_failure": "Scene assignment",
    "correct_ledger_wrong_answer": "Ledger correct, answer wrong", "incomplete_2b2": "Incomplete 2B.2",
}

sensitivity_summary = threshold_sensitivity.groupby(["threshold_scenario", "domain", "failure_flag"], dropna=False)["failure_rate"].mean().reset_index()
scenario_order = list(DIAGNOSTIC_THRESHOLD_SCENARIOS)
scenario_colors = dict(zip(scenario_order, sns.color_palette("colorblind", len(scenario_order))))
figure, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for axis, domain, domain_title in zip(axes, ["Controlled Acoustic Temporal Mapping", "Controlled Conversational Belief Updating", "Long-Horizon Conversational Navigation"], ["Acoustic mapping", "Conversational updating", "Long-horizon navigation"]):
    block = sensitivity_summary[sensitivity_summary["domain"] == domain]
    for scenario in scenario_order:
        values = block[block["threshold_scenario"] == scenario].set_index("failure_flag")["failure_rate"].reindex(stage_columns)
        axis.plot(range(len(stage_columns)), 100 * values, marker="o", linewidth=1.5, color=scenario_colors[scenario], label=scenario.title())
    axis.set_title(domain_title); axis.set_xticks(range(len(stage_columns)), [failure_flag_labels[stage] for stage in stage_columns], rotation=45, ha="right"); axis.set_ylim(0, 100); axis.set_xlabel("General diagnostic flag")
axes[0].set_ylabel("Model-balanced failure rate (%)")
handles, labels = axes[0].get_legend_handles_labels(); figure.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
figure.suptitle("General failure-flag sensitivity to declared threshold conventions", y=.99)
figure.tight_layout(rect=[0, .13, 1, .94])
save_panel_figure(figure, "panel_15", "plot_general_failure_threshold_sensitivity.png")
plt.show()

# Primary paper summary: mutually exclusive earliest-flagged-stage distributions.
first_stage_order = stage_columns + ["no applicable general flag"]
first_stage_labels = {**failure_flag_labels, "no applicable general flag": "No applicable general flag"}
first_stage_domains = ["Controlled Acoustic Temporal Mapping", "Controlled Conversational Belief Updating", "Long-Horizon Conversational Navigation"]
first_stage_domain_titles = ["Acoustic mapping", "Conversational updating", "Long-horizon navigation"]
first_stage_model_order = sorted(first_rates["model"].unique(), key=short_model_label)
stage_colors = dict(zip(first_stage_order, sns.color_palette("Set2", len(first_stage_order))))
stage_colors["no applicable general flag"] = (.75, .75, .75)
figure, axes = plt.subplots(1, 3, figsize=(15, 5.4), sharex=True, sharey=True)
for domain_index, (domain, domain_title) in enumerate(zip(first_stage_domains, first_stage_domain_titles)):
    axis = axes[domain_index]
    table = first_rates[first_rates["domain"] == domain].pivot(index="model", columns="earliest_flagged_stage", values="rate").reindex(index=first_stage_model_order, columns=first_stage_order).fillna(0)
    y_positions = np.arange(len(first_stage_model_order)); left = np.zeros(len(first_stage_model_order))
    for stage in first_stage_order:
        widths = 100 * table[stage].to_numpy(dtype=float)
        axis.barh(y_positions, widths, left=left, color=stage_colors[stage], edgecolor="white", linewidth=.35, height=.78)
        left += widths
    axis.set(title=domain_title, xlim=(0, 100), xlabel="Units (%)")
    axis.set_yticks(y_positions)
    if domain_index == 0: axis.set_yticklabels([short_model_label(model) for model in first_stage_model_order])
    else: axis.tick_params(axis="y", labelleft=False)
    axis.grid(axis="y", visible=False)
axes[0].invert_yaxis(); axes[0].set_ylabel("Model")
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=stage_colors[stage]) for stage in first_stage_order]
figure.legend(legend_handles, [first_stage_labels[stage] for stage in first_stage_order], loc="lower center", bbox_to_anchor=(.5, -.02), ncol=4, frameon=False, fontsize=8)
figure.suptitle("Earliest flagged stage by capability domain\nDiagnostic precedence; not a causal sequence", y=.99)
figure.subplots_adjust(left=.11, right=.99, bottom=.25, top=.86, wspace=.10)
save_panel_figure(figure, "panel_15", "plot_earliest_flagged_stage.png")
plt.show()

# Complete diagnostic: nonexclusive flag rates; exact denominators remain in the table above.
failure_plot_labels = {
    "protocol_or_evaluability": "Protocol /\nevaluability", "evidence_acquisition": "Evidence\nacquisition",
    "acquisition_efficiency": "Acquisition\nefficiency", "belief_construction": "Belief\nconstruction",
    "uncertainty_or_claim_support": "Uncertainty /\nsupport", "final_answer_integration": "Answer\nintegration",
    "stability_or_revision": "Stability /\nrevision", "unsupported_found_claim": "Unsupported\nFIND",
    "false_absence_claim": "Factually false\nabsence", "undercovered_true_absence_claim": "True absence\nundercovered", "scene_or_boundary_failure": "Scene /\nboundary",
    "ordering_failure": "Ordering", "scene_assignment_failure": "Scene\nassignment",
    "correct_ledger_wrong_answer": "Ledger correct,\nanswer wrong", "incomplete_2b2": "Incomplete\n2B.2",
}


def failure_block_table(domain, columns):
    selected = all_flags[(all_flags["domain"] == domain) & all_flags["failure_flag"].isin(columns)]
    values = selected.pivot(index="model", columns="failure_flag", values="failure_rate").reindex(index=first_stage_model_order, columns=columns)
    values = values.dropna(axis=1, how="all").rename(columns=failure_plot_labels)
    values.index = values.index.map(short_model_label); values.index.name = "Model"; values.columns.name = None
    return values


failure_blocks = [
    ("Controlled Acoustic Temporal Mapping", stage_columns, "General flags — acoustic mapping"),
    ("Controlled Conversational Belief Updating", stage_columns, "General flags — conversational updating"),
    ("Long-Horizon Conversational Navigation", stage_columns, "General flags — long-horizon navigation"),
    ("Long-Horizon Conversational Navigation", t2_specific, "Track 2 task-specific flags"),
]
figure, axes = plt.subplots(2, 2, figsize=(15.5, 9.2), sharey=True)
colorbar_axis = figure.add_axes([.92, .16, .018, .68])
for index, (domain, columns, title) in enumerate(failure_blocks):
    axis = axes.flat[index]; values = failure_block_table(domain, columns)
    annotations = values.applymap(lambda value: f"{100 * value:.0f}" if pd.notna(value) else "")
    sns.heatmap(100 * values, mask=values.isna(), annot=annotations, fmt="", cmap="Reds", vmin=0, vmax=100, ax=axis, cbar=index == 0, cbar_ax=colorbar_axis if index == 0 else None)
    axis.set_facecolor("#e6e6e6"); annotate_missing_heatmap_cells(axis, values)
    axis.set(title=title, xlabel="Failure flag", ylabel="Model" if index % 2 == 0 else "")
    axis.tick_params(axis="x", rotation=0, labelsize=8)
    axis.tick_params(axis="y", labelleft=index % 2 == 0)
colorbar_axis.set_ylabel("Failure rate (%)")
figure.suptitle("All applicable failure flags\nExact denominators are reported in the table above", y=.99)
figure.subplots_adjust(left=.09, right=.90, bottom=.08, top=.90, wspace=.08, hspace=.30)
save_panel_figure(figure, "panel_15", "plot_all_applicable_failure_flags.png")
plt.show()


## Supplementary trace drill-down

Representative episodes remain optional debugging and qualitative-validation assets rather than a numbered quantitative panel. They index the aggregate failure-stage results back to source files and must not be treated as independent proof.


In [ ]:
trace_columns_t1 = ["model", "track", "episode_id", "earliest_flagged_stage", "master_score", "evidence_reach_score", "temporal_belief_score", "answer_score", "source_csv"]
representative_t1 = t1_failure.sort_values(["earliest_flagged_stage", "master_score"], na_position="last").groupby("earliest_flagged_stage", dropna=False).head(1)[trace_columns_t1].head(8)
trace_columns_t2 = ["model", "family", "variant", "task_id", "earliest_flagged_stage", "master_score", "evidence_reach_score", "temporal_belief_score", "answer_score", "source_json"]
representative_t2 = t2_failure.sort_values(["earliest_flagged_stage", "master_score"], na_position="last").groupby("earliest_flagged_stage", dropna=False).head(1)[trace_columns_t2].head(8)
display(Markdown("**Track 1 trace index**")); display(percent_table(representative_t1, ["master_score", "evidence_reach_score", "temporal_belief_score", "answer_score"]))
display(Markdown("**Track 2 trace index**")); display(percent_table(representative_t2, ["master_score", "evidence_reach_score", "temporal_belief_score", "answer_score"]))
